In [ ]:
# Install pinned dependencies
!pip install -q "casadi==3.6.7" "do-mpc[full]" "stable-baselines3[extra]" gymnasium
print("Packages installed.")
print("Restart the kernel now, then Run All from the top.")

In [ ]:
# Imports and NumPy 2.x compatibility guard
import numpy as np

# CasADi's tools module
if not hasattr(np, "Inf"):
    np.Inf = np.inf
if not hasattr(np, "NaN"):
    np.NaN = np.nan
if not hasattr(np, "float_"):
    np.float_ = np.float64

import matplotlib.pyplot as plt
import time, csv
from pathlib import Path
from scipy.stats import qmc, wilcoxon, mannwhitneyu
from scipy.signal import place_poles
import gymnasium as gym
from gymnasium import spaces
from stable_baselines3 import PPO
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.env_checker import check_env
from stable_baselines3.common.callbacks import EvalCallback, BaseCallback
import builtins

import do_mpc
import casadi

# Capture the version
CASADI_VERSION = casadi.__version__
DOMPC_VERSION = do_mpc.__version__

from casadi import *

print(f"numpy {np.__version__} | casadi {CASADI_VERSION} | do-mpc {DOMPC_VERSION}")
if tuple(int(v) for v in CASADI_VERSION.split(".")[:2]) >= (3, 8):
    print("WARNING: CasADi >= 3.8 removed casadi.tools.SX and will break do-mpc.")
    print("Run the install cell above, then restart the kernel.")
print("Imports OK")

In [ ]:
# Arclength-parameterised spline path
"""Arclength-parameterized 2D cubic spline."""

import numpy as np
from scipy.integrate import quad
from scipy.interpolate import CubicSpline

class ArclengthSpline2D:
    """Arclength-parameterized natural cubic spline for."""

    def __init__(self, waypoints, filter_pts=True, min_dist=1.0, nr_resamples=10):
        """Args:."""
        wp = np.asarray(waypoints, dtype=float)
        assert wp.ndim == 2 and wp.shape[1] == 2 and len(wp) >= 3

        if filter_pts:
            wp = self._filter(wp, min_dist)

        self._spline, self.s_max = self._build(wp, nr_resamples)

    @staticmethod
    def _filter(wp, min_dist):
        chords = np.linalg.norm(np.diff(wp, axis=0), axis=1)
        if np.all(chords >= min_dist):
            return wp
        cum = np.concatenate([[0.0], np.cumsum(chords)])
        kept = [0]
        while True:
            nxt = np.searchsorted(cum, cum[kept[-1]] + min_dist, side="left")
            if nxt >= len(wp):
                break
            kept.append(nxt)
        if kept[-1] != len(wp) - 1:
            kept.append(len(wp) - 1)
        return wp[kept]

    @staticmethod
    def _build(wp, nr_resamples):
        chords = np.linalg.norm(np.diff(wp, axis=0), axis=1)
        t_knots = np.concatenate([[0.0], np.cumsum(chords)])
        t_knots /= t_knots[-1]

        p_t = CubicSpline(t_knots, wp, bc_type="natural")

        nr_segs = len(t_knots) - 1
        t_samples = np.empty(nr_segs * (nr_resamples + 1) + 1)
        for i in range(nr_segs):
            t_samples[i * (nr_resamples + 1) : (i + 1) * (nr_resamples + 1)] = (
                np.linspace(t_knots[i], t_knots[i + 1], nr_resamples + 2)[:-1]
            )
        t_samples[-1] = t_knots[-1]

        def speed(t):
            dp = p_t(t, 1)
            return np.sqrt((dp**2).sum(axis=-1))

        s_samples = np.concatenate(
            [
                [0.0],
                np.cumsum(
                    [
                        quad(speed, t_samples[i - 1], t_samples[i])[0]
                        for i in range(1, len(t_samples))
                    ]
                ),
            ]
        )
        s_max = float(s_samples[-1])

        spline = CubicSpline(s_samples, p_t(t_samples), bc_type="not-a-knot")
        return spline, s_max

    def clip(self, s):
        return np.clip(s, 0.0, self.s_max)

    def xy(self, s):
        """(x, y) at arclength s."""

        return self._spline(self.clip(s))

    def tangent(self, s):
        """Unit tangent (dx/ds, dy/ds) -."""

        return self._spline(self.clip(s), 1)

    def heading(self, s):
        """Heading angle arctan2(y', x') [rad]."""
        t = self.tangent(s)
        if np.ndim(s) == 0:
            return np.arctan2(t[1], t[0])
        return np.arctan2(t[:, 1], t[:, 0])

    def curvature(self, s):
        """Signed curvature kappa(s) = x'*y''."""
        s = self.clip(s)
        d1 = self._spline(s, 1)
        d2 = self._spline(s, 2)
        if np.ndim(s) == 0:
            return d1[0] * d2[1] - d1[1] * d2[0]
        return d1[:, 0] * d2[:, 1] - d1[:, 1] * d2[:, 0]

    def sd_to_xy(self, s, d):
        """Frenet (s, d) -> Cartesian."""
        p = self._spline(self.clip(s))
        dp = self._spline(self.clip(s), 1)
        if np.ndim(s) == 0:
            return p[0] - d * dp[1], p[1] + d * dp[0]
        return p[:, 0] - d * dp[:, 1], p[:, 1] + d * dp[:, 0]

    def xy_to_sd(self, x, y, nr_segs=50, nr_iter=10):
        """Cartesian (x, y) -> Frenet."""
        s_grid = np.linspace(0.0, self.s_max, nr_segs)
        p_grid = self._spline(s_grid)
        dists = (p_grid[:, 0] - x) ** 2 + (p_grid[:, 1] - y) ** 2
        s_est = s_grid[np.argmin(dists)]

        for _ in range(nr_iter):
            p = self._spline(s_est)
            dp = self._spline(s_est, 1)
            d2p = self._spline(s_est, 2)
            ex, ey = p[0] - x, p[1] - y
            f_d = ex * dp[0] + ey * dp[1]
            f_dd = dp[0] ** 2 + dp[1] ** 2 + ex * d2p[0] + ey * d2p[1]
            if abs(f_dd) < 1e-9:
                break
            s_est -= f_d / f_dd
            s_est = np.clip(s_est, 0.0, self.s_max)

        p = self._spline(s_est)
        dp = self._spline(s_est, 1)
        d_est = dp[0] * (y - p[1]) - dp[1] * (x - p[0])
        return s_est, d_est

print("ArclengthSpline2D defined")

In [ ]:
# Reference velocity profile (Eq. 7)
"""Reference velocity profile generator."""

import numpy as np

class ReferenceVelocityGenerator:
    def __init__(
        self,
        path,
        max_speed=8.0,
        max_lateral_acceleration=2.5,
        max_deceleration=2.5,
        nr_path_samples=100,
        filter_window=5,
    ):
        """Generate a path-aware maximum-velocity profile."""
        self.path = path
        self.max_speed = max_speed
        self.max_lateral_acceleration = max_lateral_acceleration
        self.max_deceleration = max_deceleration
        # keep at least
        self.nr_path_samples = int(max(nr_path_samples, 2 * np.ceil(self.path.s_max)))
        if filter_window > 1:
            self._filter_velocity = True
            # window size should
            window_sizes = np.array([3, 5, 7, 9], dtype=int)
            self.filter_window = window_sizes[
                (np.abs(window_sizes - filter_window)).argmin()
            ]
        else:
            self._filter_velocity = False
            self.filter_window = int(1)
        self._create_profile_lut()

    def _get_max_speed(self, s):
        """Path- and curvature-aware maximum speed."""
        # 1
        v_max_brake = np.sqrt(2.0 * (self.path.s_max - s) * self.max_deceleration)
        # clamp very low
        if v_max_brake <= 1.0:
            return 0.0
        # 2
        v_max_allowed = min(v_max_brake, self.max_speed)
        # 3
        kappa_value = abs(self.path.curvature(s))
        # ignore small curvature
        if kappa_value <= (self.max_lateral_acceleration / (v_max_allowed**2)):
            return v_max_allowed
        v_from_kappa = np.sqrt(self.max_lateral_acceleration / kappa_value)
        return min(v_from_kappa, v_max_allowed)

    def _averaging_filter(self, values):
        """Simple (moving-window) averaging filter for."""
        k = np.ones(self.filter_window, dtype=float) / self.filter_window
        # handle boundary effects
        filtered = np.convolve(values, k, mode="full")
        return filtered[self.filter_window - 1 :]

    def _create_profile_lut(self):
        """Generate points and values for."""
        # sample along arclength
        self.lut_s = np.linspace(0.0, self.path.s_max, self.nr_path_samples)
        self.lut_v = np.array([self._get_max_speed(i) for i in self.lut_s], dtype=float)
        if self._filter_velocity:
            self.lut_v = self._averaging_filter(self.lut_v)

    def get_profile(self):
        """Returns the generated velocity profile."""
        return self.lut_s, self.lut_v

    def get_maximum_speed(self, s):
        """Interpolates the maximum speed at."""
        # clip to valid
        s = self.path.clip(s)
        return np.interp(s, self.lut_s, self.lut_v)

print("ReferenceVelocityGenerator defined")

In [ ]:
# Frenet kinematic vehicle model (Eqs. 1-4)
"""A kinematic vehicle model in."""

import numpy as np

class FrenetDynamicModel:
    """FrenetDynamicModel implements kinematic vehicle dynamics."""

    def __init__(self, path, state_bounds, control_bounds, L_f=2.5, t_step=0.1):
        """Initialize the FrenetDynamicModel."""
        self.path = path
        self.state_lower = np.asarray(state_bounds[:, 0], dtype=float).reshape(-1)
        self.state_upper = np.asarray(state_bounds[:, 1], dtype=float).reshape(-1)
        self.control_lower = np.asarray(control_bounds[:, 0], dtype=float).reshape(-1)
        self.control_upper = np.asarray(control_bounds[:, 1], dtype=float).reshape(-1)
        self.L_f = L_f
        self.t_step = t_step

    def fcn_linear(self, state, control):
        """Simplified kinematic model in Frenet."""
        s, _, alpha, delta, v = state
        u1, u2 = control
        kappa = float(self.path.curvature(s))
        # dynamics
        s_dot = v
        d_dot = v * alpha
        alpha_dot = v * ((delta / self.L_f) - kappa)
        delta_dot = u1
        v_dot = u2
        return np.array([s_dot, d_dot, alpha_dot, delta_dot, v_dot])

    def fcn_nonlinear(self, state, control):
        """Extended (non-linearized) Frenet model with."""
        s, d, alpha, delta, v = state
        u1, u2 = control
        kappa = float(self.path.curvature(s))
        # avoid division by zero
        denom = 1.0 - (d * kappa)
        denom = np.copysign(max(abs(denom), 1e-6), denom)
        # dynamics
        s_dot = v * np.cos(alpha) / denom
        d_dot = v * np.sin(alpha)
        alpha_dot = (v * np.tan(delta) / self.L_f) - s_dot * kappa
        delta_dot = u1
        v_dot = u2
        return np.array([s_dot, d_dot, alpha_dot, delta_dot, v_dot])

    def _rk4_step(self, state, control, fcn):
        """Perform a single RK4 integration."""
        k1 = fcn(state, control)
        k2 = fcn(state + (self.t_step / 2) * k1, control)
        k3 = fcn(state + (self.t_step / 2) * k2, control)
        k4 = fcn(state + self.t_step * k3, control)
        return state + (self.t_step / 6) * (k1 + 2 * k2 + 2 * k3 + k4)

    def clip_state(self, state):
        """Clip state to valid bounds."""
        return np.clip(
            np.asarray(state, dtype=float).reshape(-1),
            self.state_lower,
            self.state_upper,
        )

    def clip_control(self, control):
        """Clip control input to valid."""
        return np.clip(
            np.asarray(control, dtype=float).reshape(-1),
            self.control_lower,
            self.control_upper,
        )

    def norm_state(self, state):
        """Normalize the give state vector."""

        return np.asarray(state / self.state_upper, dtype=float).reshape(-1)

    def norm_control(self, control):
        """Normalize the give conrol vector."""

        return np.asarray(control / self.control_upper, dtype=float).reshape(-1)

    def denorm_state(self, state):
        """Denormalize the give state vector."""

        return np.asarray(state * self.state_upper, dtype=float).reshape(-1)

    def denorm_control(self, control):
        """Denormalize the give conrol vector."""

        return np.asarray(control * self.control_upper, dtype=float).reshape(-1)

    def rk4_step(self, state, control, clipped=False, nonlinear=True):
        """Perform a single RK4 integration."""
        if nonlinear:
            fcn = self.fcn_nonlinear
        else:
            fcn = self.fcn_linear
        state = self._rk4_step(state, control, fcn)
        if clipped:
            state = self.clip_state(state)
        return state

print("FrenetDynamicModel defined")

In [ ]:
# VT-CPEM energy model (Eqs. 8-10)
"""EV Energy Model with Regenerative."""

import copy
import numpy as np

class EVEnergyModel:
    """Tracks net energy consumption per."""

    def __init__(self, t_step):
        # model params
        self._EV_MASS = 1500.0  # vehicle mass [kg]
        self._EV_RHO = 1.225  # air density [kg/m^3]
        self._EV_CD = 0.30  # drag coefficient [-]
        self._EV_AF = 2.20  # frontal area [m^2]
        self._EV_CR = 0.01  # rolling resistance [-]
        self._EV_G = 9.81  # gravity [m/s^2]
        self._EV_THETA = 0.0  # road gradient [rad]
        self._EV_ETA_REGEN = 0.65  # regen efficiency [-]
        self._EV_PCPU = 5.0  # computational power [W]
        self.t_step = t_step
        self.reset()

    def reset(self):
        self.e_propulsion = 0.0
        self.e_recovered = 0.0
        self.e_comp = 0.0
        self.steps = []

    def step(self, v, a, t_solve):
        """Update energy for one time."""
        no_gradient = (abs(self._EV_THETA) <= 1.0e-3)
        if no_gradient:
            f_roll = self._EV_MASS * self._EV_G * self._EV_CR
            f_gradient = 0.0
        else:
            f_roll = self._EV_MASS * self._EV_G * self._EV_CR * np.cos(self._EV_THETA)
            f_gradient = self._EV_MASS * self._EV_G * np.sin(self._EV_THETA)
        f_aero_drag = 0.5 * self._EV_RHO * self._EV_CD * self._EV_AF * v ** 2
        f_inertia = self._EV_MASS * a
        f_traction = f_aero_drag + f_roll + f_gradient + f_inertia

        # mechanical power
        P = f_traction * max(v, 0.0)

        if P >= 0.0:
            dE = P * self.t_step
            self.e_propulsion += dE
        else:
            dE = -self._EV_ETA_REGEN * abs(P) * self.t_step
            self.e_recovered += abs(dE)

        self.e_comp += self._EV_PCPU * t_solve
        self.steps.append(dE)

    @property
    def e_net(self):
        """Net energy: propulsion - recovered."""
        return self.e_propulsion - self.e_recovered + self.e_comp

    def summary(self):
        print(f"  E_propulsion : {self.e_propulsion/1000:.3f} kJ")
        print(f"  E_recovered  : {self.e_recovered/1000:.3f} kJ  (regen)")
        print(f"  E_comp       : {self.e_comp:.2f} J  (CPU)")
        print(f"  E_net        : {self.e_net/1000:.3f} kJ")
        return self.e_net

    def copy(self):
        return copy.deepcopy(self)

    def doc(self):
        print("EVEnergyModel (Electric Vehicle Energy Model)")
        print("Parameters (SI units):")
        print(f"  Vehicle Mass (M):           {self._EV_MASS} kg")
        print(f"  Air Density (rho):          {self._EV_RHO} kg/m^3")
        print(f"  Drag Coefficient (Cd):      {self._EV_CD}")
        print(f"  Frontal Area (Af):          {self._EV_AF} m^2")
        print(f"  Rolling Resistance (Cr):    {self._EV_CR}")
        print(f"  Gravity (g):                {self._EV_G} m/s^2")
        print(f"  Road Gradient (theta):      {self._EV_THETA} rad")
        print(f"  Regen Efficiency (eta):     {self._EV_ETA_REGEN}")
        print(f"  CPU Power (PCPU):           {self._EV_PCPU} W")
        print(f"  Simulation Timestep:        {self.t_step} s")

    def print(self, show_summary=True):
        self.doc()
        if show_summary:
            print("\nEnergy State Summary:")
            self.summary()

print("EVEnergyModel defined (power-sign energy split)")

In [ ]:
# NMPC controller (do-mpc / IPOPT)
"""NMPC framework for vehicle trajectory."""

import do_mpc
from casadi import *
import time

class NMPCModel:

    def __init__(self, dyn_model, vel_profile, obj_weights,
                 L_f=2.5, t_step=0.1, n_horizon=50):
        self.dyn_model = dyn_model
        self.vel_profile = vel_profile
        # defensive copy -
        self.obj_weights = dict(obj_weights)
        self.L_f = L_f
        self.t_step = t_step
        self.n_horizon = n_horizon
        self._create_model()
        self._create_optimizer()

    def _normalize_weights(self):
        """lterm = w_x * (x."""
        ub_x = self.dyn_model.state_upper
        ub_u = self.dyn_model.control_upper
        w = dict(self.obj_weights)
        w["d"]     = w["d"]     / ub_x[1] ** 2
        w["alpha"] = w["alpha"] / ub_x[2] ** 2
        w["v"]     = w["v"]     / ub_x[4] ** 2
        w["u1"]    = w["u1"]    / ub_u[0] ** 2
        w["u2"]    = w["u2"]    / ub_u[1] ** 2
        self._w = w

    def _create_model(self):
        self._model = do_mpc.model.Model("continuous")
        self._var_s     = self._model.set_variable("_x", "s")
        self._var_d     = self._model.set_variable("_x", "d")
        self._var_alpha = self._model.set_variable("_x", "alpha")
        self._var_delta = self._model.set_variable("_x", "delta")
        self._var_v     = self._model.set_variable("_x", "v")
        self._var_u1    = self._model.set_variable("_u", "u1")
        self._var_u2    = self._model.set_variable("_u", "u2")
        # kappa and v_ref
        self._tvp_kappa_ref = self._model.set_variable("_tvp", "kappa_ref")
        self._tvp_v_ref     = self._model.set_variable("_tvp", "v_ref")
        # simplified linear Frenet
        self._model.set_rhs("s", self._var_v)
        self._model.set_rhs("d", self._var_v * self._var_alpha)
        self._model.set_rhs(
            "alpha", self._var_v * ((self._var_delta / self.L_f) - self._tvp_kappa_ref)
        )
        self._model.set_rhs("delta", self._var_u1)
        self._model.set_rhs("v", self._var_u2)
        self._model.setup()

    def _create_optimizer(self):
        self._optimizer = do_mpc.controller.MPC(self._model)
        self._optimizer.settings.n_horizon = self.n_horizon
        self._optimizer.settings.t_step = self.t_step
        self._optimizer.settings.n_robust = 0
        self._optimizer.settings.store_full_solution = True
        self._optimizer.settings.supress_ipopt_output = True

        self._tvp_template_opt = self._optimizer.get_tvp_template()

        def tvp_fcn_opt(t_now):
            s_now = self.dyn_model.path.clip(float(self._optimizer.x0["s"]))
            s_end = self.dyn_model.path.clip(
                s_now + self.dyn_model.state_upper[4] * self.n_horizon * self.t_step
            )
            s_pred = np.linspace(s_now, s_end, self.n_horizon + 1)
            kappa_pred = self.dyn_model.path.curvature(s_pred)
            v_pred = self.vel_profile.get_maximum_speed(s_pred)
            for i in range(len(s_pred)):
                self._tvp_template_opt["_tvp", i] = casadi.DM(
                    [kappa_pred[i], v_pred[i]]
                )
            return self._tvp_template_opt

        self._optimizer.set_tvp_fun(tvp_fcn_opt)

        self._normalize_weights()
        lterm = (
            self._w["d"] * (self._var_d ** 2)
            + self._w["alpha"] * (self._var_alpha ** 2)
            + self._w["v"] * (self._tvp_v_ref - self._var_v) ** 2
        )
        mterm = casadi.DM(np.reshape(0, (1, 1)))
        self._optimizer.set_objective(lterm=lterm, mterm=mterm)
        self._optimizer.set_rterm(u1=self._w["u1"], u2=self._w["u2"])

        # state bounds
        for k, name in enumerate(["s", "d", "alpha", "delta", "v"]):
            self._optimizer.bounds["lower", "_x", name] = self.dyn_model.state_lower[k]
            self._optimizer.bounds["upper", "_x", name] = self.dyn_model.state_upper[k]
        # control bounds
        for k, name in enumerate(["u1", "u2"]):
            self._optimizer.bounds["lower", "_u", name] = self.dyn_model.control_lower[k]
            self._optimizer.bounds["upper", "_u", name] = self.dyn_model.control_upper[k]

        self._optimizer.settings.nlpsol_opts = {
            "ipopt.max_iter": 20,
            "ipopt.tol": 1e-6,
            "ipopt.print_level": 0,
        }
        self._optimizer.setup()

    def simulate(self, x0, ev_energy_model, warm_start=True, nr_steps=100,
                 file_name="nmpc_performance_data.npy"):
        """Closed-loop simulation on the SHARED."""
        x = np.asarray(x0, dtype=float).reshape(-1)
        self._optimizer.x0 = x
        self._optimizer.set_initial_guess()

        states = np.zeros((nr_steps + 1, 5))
        controls = np.zeros((nr_steps, 2))
        times_exe = np.zeros((nr_steps, 1))
        states[0] = x.copy()
        failed = False

        for i in range(nr_steps):
            if not warm_start:
                self._optimizer.set_initial_guess()
            # timing
            t0 = time.perf_counter()
            u0 = self._optimizer.make_step(x.reshape(-1, 1))
            t_exe = time.perf_counter() - t0
            u = np.asarray(u0, dtype=float).reshape(-1)
            # shared nonlinear
            x = self.dyn_model.rk4_step(x, u, clipped=True, nonlinear=True)

            states[i + 1] = x.copy()
            controls[i] = u.copy()
            times_exe[i] = t_exe
            ev_energy_model.step(float(x[4]), float(u[1]), t_exe)

            if float(x[0]) >= (self.dyn_model.path.s_max - 0.1):
                print(f"End of path at step {i+1}")
                states = states[: i + 2]
                controls = controls[: i + 1]
                times_exe = times_exe[: i + 1]
                break
            if abs(float(x[1])) >= abs(self.dyn_model.state_upper[1]) - 0.1:
                print(f"Off-road at step {i+1} (d={x[1]:.3f} m) - FAILED")
                failed = True
                states = states[: i + 2]
                controls = controls[: i + 1]
                times_exe = times_exe[: i + 1]
                break
            if (i + 1) % 30 == 0:
                print(
                    f"  Step {i+1:3d} | s={x[0]:.1f}m | d={x[1]:.3f}m | "
                    f"v={x[4]:.2f}m/s | solve={t_exe*1000:.1f}ms"
                )

        if file_name is not None:
            np.save(file_name, times_exe)

        print(
            f"\n NMPC Done | Avg: {times_exe.mean()*1000:.1f}ms | "
            f"p95: {np.percentile(times_exe, 95)*1000:.1f}ms | "
            f"Max: {times_exe.max()*1000:.1f}ms | Steps: {len(times_exe)}"
        )
        print("Energy:")
        ev_energy_model.print()
        return states, controls, times_exe, failed

print("NMPCModel defined (shared nonlinear plant, safe weight normalization)")

In [ ]:
# PID-SF: gain-scheduled full-state feedback
# Gain-Scheduled State Feedback Controller (Ackermann).

import time
import numpy as np
from scipy.signal import place_poles

class PIDModel:

    V_NOM = 5.0
    DESIRED_POLES = np.array([-1.5, -2.5, -5.0])

    def __init__(self, dyn_model, vel_profile, L_f=4.0, t_step=0.1,
                 Kff_kappa=1.0, Kp_lon=1.0, Ki_lon=0.02):
        self.dyn_model = dyn_model
        self.vel_profile = vel_profile
        self.L_f = L_f
        self.t_step = t_step
        self.Kff_kappa = Kff_kappa
        self.Kp_lon = Kp_lon
        self.Ki_lon = Ki_lon

        self._Kd_nom, self._Kalpha_nom, self._Kdelta = \
            self._compute_gains(self.V_NOM, self.L_f, self.DESIRED_POLES)
        self._lon_int = 0.0

    @staticmethod
    def _compute_gains(v, L_f, poles):
        A = np.array([[0, v, 0],
                      [0, 0, v / L_f],
                      [0, 0, 0]])
        B = np.array([[0], [0], [1]])
        K = place_poles(A, B, poles).gain_matrix[0]
        return float(K[0]), float(K[1]), float(K[2])

    def _lateral_gains(self, v):
        """Speed-scheduled gains keeping the closed-loop."""
        v = max(v, 0.5)  # v_min clamp (avoids)
        r = self.V_NOM / v
        return (self._Kd_nom * r ** 2,
                self._Kalpha_nom * r,
                self._Kdelta)

    def reset(self):
        self._lon_int = 0.0

    def compute_control(self, state):
        s, d, alpha, delta, v = np.asarray(state, dtype=float)
        v = max(v, 0.1)
        s_c = float(self.dyn_model.path.clip(s))

        kappa = float(self.dyn_model.path.curvature(s_c))
        Kd, Ka, Kdt = self._lateral_gains(v)

        u1 = (- Kd * d
              - Ka * alpha
              - Kdt * delta
              + self.Kff_kappa * kappa * self.L_f)

        v_ref = float(self.vel_profile.get_maximum_speed(s_c))
        e_v = v_ref - v
        self._lon_int = np.clip(self._lon_int + e_v * self.t_step, -5.0, 5.0)
        u2 = self.Kp_lon * e_v + self.Ki_lon * self._lon_int

        return self.dyn_model.clip_control(np.array([u1, u2]))

    def simulate(self, x0, ev_energy_model, nr_steps=200, file_name=None):
        """Closed-loop simulation on the shared."""
        self.reset()
        x0 = self.dyn_model.clip_state(np.asarray(x0, dtype=float))

        states = np.zeros((nr_steps + 1, 5))
        controls = np.zeros((nr_steps, 2))
        times_exe = np.zeros((nr_steps, 1))
        states[0] = x0.copy()
        failed = False

        for i in range(nr_steps):
            t_start = time.perf_counter()
            u = self.compute_control(x0)
            t_exe = time.perf_counter() - t_start

            x0 = self.dyn_model.rk4_step(x0, u, clipped=True, nonlinear=True)

            states[i + 1] = x0.copy()
            controls[i] = u.copy()
            times_exe[i] = t_exe
            ev_energy_model.step(float(x0[4]), float(u[1]), t_exe)

            s_now = float(x0[0])
            d_now = abs(float(x0[1]))

            if s_now >= self.dyn_model.path.s_max - 0.1:
                print(f"End of path at step {i + 1}")
                states = states[:i + 2]
                controls = controls[:i + 1]
                times_exe = times_exe[:i + 1]
                break
            if d_now >= abs(self.dyn_model.state_upper[1]) - 0.1:
                print(f"Off-road at step {i + 1}  (d={x0[1]:.3f} m) - FAILED")
                failed = True
                states = states[:i + 2]
                controls = controls[:i + 1]
                times_exe = times_exe[:i + 1]
                break

        if file_name is not None:
            np.save(file_name, times_exe)

        print(
            f"\n PIDModel Done | Avg: {times_exe.mean()*1e3:.3f} ms | "
            f"Max: {times_exe.max()*1e3:.3f} ms | Steps: {len(times_exe)}"
        )
        print("Energy:")
        ev_energy_model.summary()
        return states, controls, times_exe, failed

print("PIDModel defined (pole-invariant scheduling, perf_counter timing)")

In [ ]:
# PPO environment: preview observation + energy reward
class PathFollowEnv(gym.Env):
    metadata = {"render_modes": []}

    PREVIEW_DISTS = (5.0, 15.0, 30.0)  # metres ahead

    # VT-CPEM flat-road parameters
    _EM_M, _EM_RHO, _EM_CD, _EM_AF = 1500.0, 1.225, 0.30, 2.20
    _EM_CR, _EM_G, _EM_ETA = 0.01, 9.81, 0.65
    _EM_P_SCALE = _EM_M * 2.5 * 8.0  # 30 kW scale

    # Per-phase reward weights
    _WEIGHTS = {
        0: dict(w_cte=0.3, w_heading=0.3, w_prog=1.0,
                w_delta=0.02, w_v=0.8, w_acc=0.10, w_energy=0.0),
        1: dict(w_cte=1.2, w_heading=0.8, w_prog=0.5,
                w_delta=0.05, w_v=0.3, w_acc=0.05, w_energy=0.0),
        2: dict(w_cte=1.2, w_heading=0.8, w_prog=0.8,
                w_delta=0.05, w_v=0.3, w_acc=0.05, w_energy=0.0),
        3: dict(w_cte=1.5, w_heading=1.0, w_prog=0.5,
                w_delta=0.10, w_v=0.4, w_acc=0.05, w_energy=0.0),
        4: dict(w_cte=1.0, w_heading=0.6, w_prog=0.5,
                w_delta=0.10, w_v=0.5, w_acc=0.15, w_energy=0.3),
    }

    def __init__(self, dyn_model, vel_profile, phase=1, t_step=0.1,
                 preview=False, energy_reward=False):
        super().__init__()
        self.dyn_model = dyn_model
        self.vel_profile = vel_profile
        self.phase = phase
        self.t_step = t_step
        self.preview = bool(preview)
        self.energy_reward = bool(energy_reward)

        self.s_max = float(dyn_model.path.s_max)
        self.d_max = float(dyn_model.state_upper[1])
        self.v_max = float(dyn_model.state_upper[4])
        self.u1_max = float(dyn_model.control_upper[0])
        self.u2_max = float(dyn_model.control_upper[1])

        obs_dim = 8 + (2 * len(self.PREVIEW_DISTS) if self.preview else 0)
        self.observation_space = spaces.Box(-1., 1., shape=(obs_dim,),
                                            dtype=np.float32)
        self.action_space = spaces.Box(-1., 1., shape=(2,), dtype=np.float32)

        self._dyn_ph1 = None
        self._vel_ph1 = None
        self._dyn_orig = dyn_model
        self._vel_orig = vel_profile
        self._train_noise = False  # enabled in Phase
        self._noise_sigma = 0.02  # sigma for domain
        self.state = None
        self._steps = 0
        self._MAX = 500

    # Internal helpers

    def _clip_obs(self, obs):
        return np.clip(obs.astype(np.float32), -1., 1.)

    def _traction_power(self, v, a):
        """VT-CPEM flat-road mechanical power P."""
        v = max(float(v), 0.0)
        f = (self._EM_M * float(a)
             + 0.5 * self._EM_RHO * self._EM_CD * self._EM_AF * v ** 2
             + self._EM_M * self._EM_G * self._EM_CR)
        return f * v

    def _obs(self):
        s, d, alpha, delta, v = self.state
        s_c = float(np.clip(s, 0., self.s_max))
        kappa = float(self.dyn_model.path.curvature(s_c))
        v_ref = float(self.vel_profile.get_maximum_speed(s_c))
        base = [
            d / self.d_max,
            alpha / np.pi,
            np.clip(kappa / KAPPA_SCALE, -1., 1.),  # fixed scale, all
            v_ref / self.v_max,
            delta / DELTA_MAX,
            v / self.v_max,
            s_c / self.s_max,
            (v - v_ref) / self.v_max,
        ]
        if self.preview:
            for ds_p in self.PREVIEW_DISTS:
                s_p = float(np.clip(s_c + ds_p, 0., self.s_max))
                base.append(np.clip(
                    float(self.dyn_model.path.curvature(s_p)) / KAPPA_SCALE,
                    -1., 1.))
            for ds_p in self.PREVIEW_DISTS:
                s_p = float(np.clip(s_c + ds_p, 0., self.s_max))
                base.append(
                    float(self.vel_profile.get_maximum_speed(s_p)) / self.v_max)
        obs = np.array(base)
        # Phase-3 domain randomization
        if self._train_noise and self._noise_sigma > 0.0:
            obs = obs + self.np_random.normal(0., self._noise_sigma,
                                              size=obs.shape)
        return self._clip_obs(obs)

    def _denorm(self, action):
        return self.dyn_model.clip_control(np.array([
            float(action[0]) * self.u1_max,
            float(action[1]) * self.u2_max,
        ]))

    def _reward(self, s_prev, u):
        w = self._WEIGHTS.get(self.phase, self._WEIGHTS[1])
        s, d, alpha, delta, v = self.state
        s_c = float(np.clip(s, 0., self.s_max))
        v_ref = float(self.vel_profile.get_maximum_speed(s_c))

        v_along = max(float(v), 0.) * np.cos(float(alpha))
        r_cth = (w["w_heading"] * v_along / max(self.v_max, 1e-3)
                 - w["w_cte"] * abs(float(d)) / self.d_max)

        max_step = self.v_max * self.t_step
        ds = max(float(s) - float(s_prev), 0.)
        r_prog = w["w_prog"] * float(np.clip(ds / max(max_step, 1e-6), 0., 1.))

        p_delta = w["w_delta"] * abs(float(delta)) / DELTA_MAX
        p_v = w["w_v"] * abs(float(v) - v_ref) / max(self.v_max, 1e-3)
        p_acc = w["w_acc"] * abs(float(u[1])) / self.u2_max

        if self.energy_reward:
            # model-in-the-loop energy term
            P = self._traction_power(v, u[1])
            e_norm = (max(P, 0.) - self._EM_ETA * max(-P, 0.)) / self._EM_P_SCALE
            p_energy = w["w_energy"] * e_norm  # can be NEGATIVE
        else:
            p_energy = w["w_energy"] * abs(float(u[1])) / self.u2_max

        r = r_cth + r_prog - p_delta - p_v - p_acc - p_energy

        # extra speed-error penalty
        if abs(float(v) - v_ref) / max(self.v_max, 1e-3) > 0.25:
            r -= 0.5

        return float(r)

    def _sample_ic(self):
        rng = self.np_random
        if self.phase == 0:
            return np.array([
                rng.uniform(0., self.s_max * 0.5),
                rng.uniform(-0.1, 0.1),
                rng.uniform(-0.02, 0.02),
                0.0,
                rng.uniform(1.0, self.v_max * 0.7),
            ])
        elif self.phase == 1:
            return np.array([
                rng.uniform(0., self.s_max * 0.3),
                rng.uniform(-0.3, 0.3),
                rng.uniform(-0.05, 0.05),
                0.0,
                rng.uniform(2.0, self.v_max * 0.6),
            ])
        elif self.phase == 2:
            return np.array([
                0.0,
                rng.uniform(-0.5, 0.5),
                rng.uniform(-0.10, 0.10),
                0.0,
                rng.uniform(2.0, self.v_max * 0.8),
            ])
        elif self.phase == 3:
            return np.array([
                rng.uniform(0., self.s_max * 0.4),
                rng.uniform(-1.5, 1.5),
                rng.uniform(-0.20, 0.20),
                0.0,
                rng.uniform(1.0, self.v_max),
            ])
        else:  # phase 4
            return np.array([
                rng.uniform(0., self.s_max * 0.5),
                rng.uniform(-1.0, 1.0),
                rng.uniform(-0.15, 0.15),
                0.0,
                rng.uniform(2.0, self.v_max),
            ])

    # Gymnasium interface

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.state = self.dyn_model.clip_state(self._sample_ic())
        self._steps = 0
        return self._obs(), {}

    def step(self, action):
        u = self._denorm(action)
        s_prev = float(self.state[0])

        self.state = self.dyn_model.rk4_step(
            self.state, u, clipped=True, nonlinear=True)
        self._steps += 1

        r = self._reward(s_prev, u)

        s, d = float(self.state[0]), float(self.state[1])
        oob = abs(d) >= self.d_max - 0.1
        terminated = bool(s >= self.s_max - 0.1 or oob)
        truncated = bool(self._steps >= self._MAX)

        if oob:
            r -= 5.0  # single terminal off-road

        return self._obs(), r, terminated, truncated, {"failed": oob}

    def set_phase(self, phase):
        assert phase in (0, 1, 2, 3, 4), f"phase must be 0-4, got {phase}"
        self.phase = phase
        self._train_noise = (phase == 3)
        if phase == 0 and self._dyn_ph1 is not None:
            self.dyn_model = self._dyn_ph1
            self.vel_profile = self._vel_ph1
            self.s_max = float(self._dyn_ph1.path.s_max)
        elif phase >= 1 and self._dyn_ph1 is not None:
            self.dyn_model = self._dyn_orig
            self.vel_profile = self._vel_orig
            self.s_max = float(self._dyn_orig.path.s_max)

print("PathFollowEnv defined (preview obs + VT-CPEM energy reward, 2x2-ready)")

In [ ]:
# Paths, model instances and constants
D_MAX     = 5.0
ALPHA_MAX = np.pi
DELTA_MAX = 0.6
V_MAX     = 8.0
A_LAT_MAX = 2.5
A_LNG_MAX = 3.0
U1_MAX    = 0.4
U2_MAX    = 2.5
L_F       = 4.0
T_SMPL    = 0.1
N_HORIZON = 50

# Path
wx = np.array([0, 10, 20, 30, 40, 50], dtype=float)
wy = np.array([0,  3,  0, -3,  0,  0], dtype=float)
path = ArclengthSpline2D(
    np.column_stack([wx, wy]), filter_pts=False, nr_resamples=20
)
print(f"Path built | s_max = {path.s_max:.2f} m")

# Velocity profile
vel_profile = ReferenceVelocityGenerator(
    path,
    max_speed=V_MAX,
    max_lateral_acceleration=A_LAT_MAX,
    max_deceleration=A_LNG_MAX,
    nr_path_samples=100,
    filter_window=3,
)
s_t = np.linspace(0, path.s_max, 200)
v_t = vel_profile.get_maximum_speed(s_t)
print(f"v_ref range: [{v_t.min():.2f}, {v_t.max():.2f}] m/s")

# Dynamic model
state_bounds   = np.array([
    [0, path.s_max],
    [-D_MAX,    D_MAX],
    [-ALPHA_MAX, ALPHA_MAX],
    [-DELTA_MAX, DELTA_MAX],
    [0,         V_MAX],
])
control_bounds = np.array([[-U1_MAX, U1_MAX], [-U2_MAX, U2_MAX]])
dyn_model = FrenetDynamicModel(
    path=path,
    state_bounds=state_bounds,
    control_bounds=control_bounds,
    L_f=L_F,
    t_step=T_SMPL,
)
print("Dynamic model ready")

# Initial condition
X0 = np.array([0.0, 0.5, 0.1, 0.0, 3.0])
# Long straight
WP_STRAIGHT = np.array([[0,0],[50,0],[100,0],[150,0],[200,0]], dtype=float)
path_straight = ArclengthSpline2D(
    WP_STRAIGHT, filter_pts=False, nr_resamples=10
)
print(f"Straight path: s_max={path_straight.s_max:.1f}m")

# ISO 3888-1
def _iso3888_1_centerline(offset=3.5, ds=2.5):
    xs, ys = [], []
    # section 1
    for x in np.arange(0.0, 15.0, ds):
        xs.append(x); ys.append(0.0)
    # section 2
    for x in np.arange(15.0, 45.0, ds):
        t = (x - 15.0) / 30.0
        xs.append(x); ys.append(offset * 0.5 * (1 - np.cos(np.pi * t)))
    # section 3
    for x in np.arange(45.0, 70.0, ds):
        xs.append(x); ys.append(offset)
    # section 4
    for x in np.arange(70.0, 95.0, ds):
        t = (x - 70.0) / 25.0
        xs.append(x); ys.append(offset * 0.5 * (1 + np.cos(np.pi * t)))
    # sections 5+6
    for x in np.arange(95.0, 125.0 + ds, ds):
        xs.append(x); ys.append(0.0)
    return np.column_stack([xs, ys])

WP_LC = _iso3888_1_centerline()
path_lc = ArclengthSpline2D(WP_LC, filter_pts=False, nr_resamples=10)
print(f"ISO 3888-1 lane-change: s_max={path_lc.s_max:.2f}m (nominal 125 m)")

# kappa_max, S-curve
s_kappa_check = np.linspace(0, path.s_max, 500)
KAPPA_MAX = float(np.max(np.abs(path.curvature(s_kappa_check))))
KAPPA_SCALE = KAPPA_MAX
print(f"kappa_max (S-curve): {KAPPA_MAX:.4f} m-1  ->  KAPPA_SCALE = {KAPPA_SCALE:.4f}")

# Chicane path
WP_CHICANE = np.array([
    [0,0],[20,0],[35,5],[50,10],[65,10],[80,5],[90,0],[110,0]
], dtype=float)
path_chicane = ArclengthSpline2D(
    WP_CHICANE, filter_pts=False, nr_resamples=20)
print(f"Chicane:       s_max={path_chicane.s_max:.2f}m")

In [ ]:
# Environment validation
_test_env = PathFollowEnv(dyn_model, vel_profile, phase=1, t_step=T_SMPL)
check_env(_test_env, warn=True)
print("Environment check passed")

# Quick sanity
obs, _ = _test_env.reset(seed=0)
act    = _test_env.action_space.sample()
obs2, r, done, trunc, _ = _test_env.step(act)
print(f"  obs shape : {obs.shape}  (should be (8,))")
print(f"  obs range : [{obs.min():.2f}, {obs.max():.2f}]  (should be in [-1,1])")
print(f"  reward    : {r:.4f}")
print(f"  action range: {_test_env.action_space.low}  {_test_env.action_space.high}")
del _test_env

In [ ]:
# NMPC benchmark on the S-curve
obj_weights = {
    "d"    : 10.0,
    "alpha": 100.0,
    "v"    : 1.0,
    "u1"   : 1.0,
    "u2"   : 1.0,
}
mpc = NMPCModel(
    dyn_model=dyn_model,
    vel_profile=vel_profile,
    obj_weights=obj_weights,
    L_f=L_F, t_step=T_SMPL, n_horizon=N_HORIZON,
)
ev_nmpc = EVEnergyModel(t_step=T_SMPL)
states_nmpc, controls_nmpc, times_nmpc, fail_nmpc = mpc.simulate(
    x0=X0.copy(), ev_energy_model=ev_nmpc,
    warm_start=True, nr_steps=200,
)

# Metrics
rmse_d_nmpc = np.sqrt(np.mean(states_nmpc[1:,1]**2))
rmse_a_nmpc = np.sqrt(np.mean(states_nmpc[1:,2]**2))
print(f"\nNMPC | RMSE_d={rmse_d_nmpc:.4f}m | RMSE_alpha={rmse_a_nmpc:.4f}rad")
print(f"      Avg solve={times_nmpc.mean()*1000:.1f}ms")

# Plot
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
s_plt = np.linspace(0, path.s_max, 300)
xy_ref  = path.xy(s_plt)
xy_nmpc = np.array([path.sd_to_xy(s,d) for s,d in zip(states_nmpc[:,0],states_nmpc[:,1])])

axes[0].plot(xy_ref[:,0], xy_ref[:,1], 'b--', lw=2, label='Reference')
axes[0].plot(xy_nmpc[:,0], xy_nmpc[:,1], 'r-', lw=2, label='NMPC')
axes[0].set_aspect('equal'); axes[0].legend(); axes[0].grid(True)
axes[0].set_title('Trajectory')

t_vec = np.arange(len(states_nmpc)) * T_SMPL
axes[1].plot(t_vec, states_nmpc[:,1], 'b-', lw=2)
axes[1].axhline(0, color='k', ls='--'); axes[1].grid(True)
axes[1].set_title('Lateral Deviation d [m]'); axes[1].set_xlabel('t [s]')

axes[2].plot(times_nmpc*1000, 'purple', lw=1.5)
axes[2].axhline(100, color='r', ls='--', lw=1, label='100ms')
axes[2].set_title('Solve Time [ms]'); axes[2].legend(); axes[2].grid(True)
plt.suptitle('NMPC Results', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# NMPC warm- versus cold-start
ev_cold = EVEnergyModel(t_step=T_SMPL)
ev_warm = EVEnergyModel(t_step=T_SMPL)

st_cold, _, tm_cold, _ = mpc.simulate(
    x0=X0.copy(), ev_energy_model=ev_cold,
    warm_start=False, nr_steps=20, file_name=None,
)
st_warm, _, tm_warm, _ = mpc.simulate(
    x0=X0.copy(), ev_energy_model=ev_warm,
    warm_start=True,  nr_steps=20, file_name=None,
)

print(f"Cold: avg={tm_cold.mean()*1000:.1f}ms | RMSE_d={np.sqrt(np.mean(st_cold[1:,1]**2)):.4f}m")
print(f"Warm: avg={tm_warm.mean()*1000:.1f}ms | RMSE_d={np.sqrt(np.mean(st_warm[1:,1]**2)):.4f}m")
print(f"Speedup: {tm_cold.mean()/tm_warm.mean():.2f}x")

In [ ]:
# PID-SF benchmark 
pid = PIDModel(
    dyn_model=dyn_model,
    vel_profile=vel_profile,
    L_f=L_F,
    t_step=T_SMPL,
    Kff_kappa=1.0,
    Kp_lon=1.0,
    Ki_lon=0.02,  # value used everywhere
)

# Print computed gains
v_nom = PIDModel.V_NOM
Kd, Ka, Kdt = pid._Kd_nom, pid._Kalpha_nom, pid._Kdelta
print(f"State feedback gains at v={v_nom} m/s:")
print(f"  Kd={Kd:.3f}, Kalpha={Ka:.3f}, Kdelta={Kdt:.3f}")
print(f"  Closed-loop poles: {sorted(PIDModel.DESIRED_POLES)}")
print()

ev_pid = EVEnergyModel(t_step=T_SMPL)
states_pid, controls_pid, times_pid, fail_pid = pid.simulate(
    x0=X0.copy(),  # same IC as
    ev_energy_model=ev_pid,
    nr_steps=200,
)

rmse_d_pid = np.sqrt(np.mean(states_pid[1:,1]**2))
rmse_a_pid = np.sqrt(np.mean(states_pid[1:,2]**2))
print(f"\nPID | RMSE_d={rmse_d_pid:.4f}m | RMSE_alpha={rmse_a_pid:.4f}rad")
print(f"     Avg solve={times_pid.mean()*1000:.3f}ms | Steps={len(times_pid)}")

# Comparison plot
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
s_plt  = np.linspace(0, path.s_max, 300)
xy_ref = path.xy(s_plt)
xy_pid = np.array([path.sd_to_xy(s,d)
                   for s,d in zip(states_pid[:,0], states_pid[:,1])])
xy_nmpc_local = np.array([path.sd_to_xy(s,d)
                           for s,d in zip(states_nmpc[:,0], states_nmpc[:,1])])

axes[0].plot(xy_ref[:,0], xy_ref[:,1], 'k--', lw=2, label='Reference')
axes[0].plot(xy_nmpc_local[:,0], xy_nmpc_local[:,1], 'r-', lw=2, label='NMPC')
axes[0].plot(xy_pid[:,0], xy_pid[:,1],  'b-', lw=2, label='PID (SF)')
axes[0].set_aspect('equal'); axes[0].legend(); axes[0].grid(True)
axes[0].set_title('Trajectory')

t_n = np.arange(len(states_nmpc)) * T_SMPL
t_p = np.arange(len(states_pid))  * T_SMPL
axes[1].plot(t_n, states_nmpc[:,1], 'r-', lw=2, label='NMPC')
axes[1].plot(t_p, states_pid[:,1],  'b-', lw=2, label='PID (SF)')
axes[1].axhline(0, color='k', ls='--')
axes[1].legend(); axes[1].grid(True)
axes[1].set_title('Lateral Deviation d [m]'); axes[1].set_xlabel('t [s]')

axes[2].bar(['NMPC', 'PID (SF)'],
            [np.sqrt(np.mean(states_nmpc[1:,1]**2)), rmse_d_pid],
            color=['red','blue'], alpha=0.75, edgecolor='k')
axes[2].set_title('RMSE_d Comparison [m]')
axes[2].set_ylabel('[m]'); axes[2].grid(True, axis='y')

plt.suptitle('NMPC vs PID (State Feedback)', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# Stanley geometric baseline 
class StanleyModel:

    def __init__(self, dyn_model, vel_profile, L_f=4.0, t_step=0.1,
                 k_e=1.0, k_soft=1.0, K_delta=5.0, Kff_kappa=1.0,
                 Kp_lon=1.0, Ki_lon=0.02):
        self.dyn_model = dyn_model
        self.vel_profile = vel_profile
        self.L_f = L_f
        self.t_step = t_step
        self.k_e = k_e
        self.k_soft = k_soft
        self.K_delta = K_delta
        self.Kff_kappa = Kff_kappa
        self.Kp_lon = Kp_lon
        self.Ki_lon = Ki_lon
        self._lon_int = 0.0

    def reset(self):
        self._lon_int = 0.0

    def compute_control(self, state):
        s, d, alpha, delta, v = np.asarray(state, dtype=float)
        v = max(v, 0.1)
        s_c = float(self.dyn_model.path.clip(s))
        kappa = float(self.dyn_model.path.curvature(s_c))

        delta_des = (- alpha
                     - np.arctan(self.k_e * d / (self.k_soft + v))
                     + self.Kff_kappa * kappa * self.L_f)
        delta_des = float(np.clip(delta_des,
                                  self.dyn_model.state_lower[3],
                                  self.dyn_model.state_upper[3]))
        u1 = self.K_delta * (delta_des - delta)

        v_ref = float(self.vel_profile.get_maximum_speed(s_c))
        e_v = v_ref - v
        self._lon_int = np.clip(self._lon_int + e_v * self.t_step, -5.0, 5.0)
        u2 = self.Kp_lon * e_v + self.Ki_lon * self._lon_int

        return self.dyn_model.clip_control(np.array([u1, u2]))

    def simulate(self, x0, ev_energy_model, nr_steps=200, file_name=None):
        self.reset()
        x0 = self.dyn_model.clip_state(np.asarray(x0, dtype=float))
        states = np.zeros((nr_steps + 1, 5))
        controls = np.zeros((nr_steps, 2))
        times_exe = np.zeros((nr_steps, 1))
        states[0] = x0.copy()
        failed = False
        for i in range(nr_steps):
            t_start = time.perf_counter()
            u = self.compute_control(x0)
            t_exe = time.perf_counter() - t_start
            x0 = self.dyn_model.rk4_step(x0, u, clipped=True, nonlinear=True)
            states[i + 1] = x0.copy()
            controls[i] = u.copy()
            times_exe[i] = t_exe
            ev_energy_model.step(float(x0[4]), float(u[1]), t_exe)
            if float(x0[0]) >= self.dyn_model.path.s_max - 0.1:
                print(f"End of path at step {i + 1}")
                states = states[:i + 2]; controls = controls[:i + 1]
                times_exe = times_exe[:i + 1]
                break
            if abs(float(x0[1])) >= abs(self.dyn_model.state_upper[1]) - 0.1:
                print(f"Off-road at step {i + 1} (d={x0[1]:.3f} m) - FAILED")
                failed = True
                states = states[:i + 2]; controls = controls[:i + 1]
                times_exe = times_exe[:i + 1]
                break
        if file_name is not None:
            np.save(file_name, times_exe)
        print(f"\n Stanley Done | Avg: {times_exe.mean()*1e3:.3f} ms | "
              f"Steps: {len(times_exe)}")
        ev_energy_model.summary()
        return states, controls, times_exe, failed

# Run Stanley
stanley = StanleyModel(dyn_model=dyn_model, vel_profile=vel_profile,
                       L_f=L_F, t_step=T_SMPL,
                       k_e=1.0, k_soft=1.0, K_delta=5.0,
                       Kff_kappa=1.0, Kp_lon=1.0, Ki_lon=0.02)
ev_stan = EVEnergyModel(t_step=T_SMPL)
states_stan, controls_stan, times_stan, fail_stan = stanley.simulate(
    x0=X0.copy(), ev_energy_model=ev_stan, nr_steps=200)

rmse_d_stan = np.sqrt(np.mean(states_stan[1:, 1] ** 2))
print(f"\nStanley | RMSE_d={rmse_d_stan:.4f}m | "
      f"Avg solve={times_stan.mean()*1000:.3f}ms")

In [ ]:
# PPO training: 2x2 ablation, then winner x 10 seeds
class LearningCurveCallback(BaseCallback):
    """Records mean episode reward every."""
    def __init__(self, eval_freq=5000, verbose=0):
        super().__init__(verbose)
        self.eval_freq = eval_freq
        self.rewards = []
        self.steps = []
        self._ep_buf = []

    def _on_step(self):
        infos = self.locals.get('infos', [])
        for info in infos:
            if 'episode' in info:
                self._ep_buf.append(info['episode']['r'])
        if self.num_timesteps % self.eval_freq == 0 and self._ep_buf:
            self.rewards.append(float(np.mean(self._ep_buf[-20:])))
            self.steps.append(self.num_timesteps)
            self._ep_buf = []
        return True

# Phase-0 straight
state_bounds_straight = np.array([
    [0, path_straight.s_max],
    [-D_MAX, D_MAX],
    [-ALPHA_MAX, ALPHA_MAX],
    [-DELTA_MAX, DELTA_MAX],
    [0, V_MAX],
])
dyn_straight = FrenetDynamicModel(
    path=path_straight, state_bounds=state_bounds_straight,
    control_bounds=control_bounds, L_f=L_F, t_step=T_SMPL)
vel_straight = ReferenceVelocityGenerator(
    path_straight, max_speed=V_MAX,
    max_lateral_acceleration=A_LAT_MAX,
    max_deceleration=A_LNG_MAX,
    nr_path_samples=100, filter_window=3)
print("Straight path dynamic model ready")

FAST_TEST = False

PHASE_STEPS = (
    {0: 1_000, 1: 2_000, 2: 2_000, 3: 1_500, 4: 1_500}
    if FAST_TEST else
    {0: 50_000, 1: 120_000, 2: 80_000, 3: 80_000, 4: 120_000}  # 450k
)

PPO_KWARGS = dict(
    policy="MlpPolicy",
    n_steps=2048,
    batch_size=256,
    n_epochs=10,
    gamma=0.99,
    gae_lambda=0.95,
    clip_range=0.2,
    ent_coef=0.005,
    vf_coef=0.5,
    max_grad_norm=0.5,
    learning_rate=3e-4,
    policy_kwargs=dict(net_arch=[256, 256]),
    verbose=0,
    device="cpu",
)

SEEDS = [42] if FAST_TEST else [42, 43, 44, 45, 46, 47, 48, 49, 50, 51]
PRIMARY_SEED = 42

def train_ppo_curriculum(seed, preview=False, energy_reward=False,
                         w_energy_ph4=None, callback=None):
    """Full 5-phase curriculum with the."""
    env = Monitor(PathFollowEnv(dyn_model, vel_profile, phase=1,
                                t_step=T_SMPL, preview=preview,
                                energy_reward=energy_reward))
    env.unwrapped._dyn_ph1 = dyn_straight
    env.unwrapped._vel_ph1 = vel_straight
    env.unwrapped._dyn_orig = dyn_model
    env.unwrapped._vel_orig = vel_profile
    if w_energy_ph4 is not None:
        env.unwrapped._WEIGHTS = {
            **PathFollowEnv._WEIGHTS,
            4: {**PathFollowEnv._WEIGHTS[4], "w_energy": float(w_energy_ph4)},
        }

    kwargs = dict(PPO_KWARGS)
    kwargs["seed"] = seed
    model = PPO(env=env, **kwargs)
    for phase in (0, 1, 2, 3, 4):
        n = int(PHASE_STEPS[phase])
        env.unwrapped.set_phase(phase)
        print(f"    Phase {phase} ({n:,} steps)...", flush=True)
        model.learn(total_timesteps=n,
                    reset_num_timesteps=(phase == 0),
                    callback=callback,
                    progress_bar=False)
    return model

def eval_policy_scurve(model, x0, preview=False, energy_reward=False,
                       max_steps=500):
    """Deterministic rollout on the S-curve."""
    env = PathFollowEnv(dyn_model, vel_profile, phase=2, t_step=T_SMPL,
                        preview=preview, energy_reward=energy_reward)
    env.reset(seed=0)
    env.state = dyn_model.clip_state(x0.copy())
    obs = env._obs()
    st = [env.state.copy()]
    ev = EVEnergyModel(t_step=T_SMPL)
    fail = False
    for _ in range(max_steps):
        act, _ = model.predict(obs, deterministic=True)
        obs, _, term, trunc, info = env.step(act)
        u = env._denorm(act)
        ev.step(float(env.state[4]), float(u[1]), 0.)
        st.append(env.state.copy())
        fail = fail or bool(info.get("failed", False))
        if term or trunc or float(env.state[0]) >= path.s_max - 0.1:
            break
    arr = np.array(st)
    rmse = float(np.sqrt(np.mean(arr[1:, 1] ** 2))) if len(arr) > 1 else float("nan")
    regen = 100. * ev.e_recovered / max(ev.e_propulsion, 1e-9)
    return rmse, float(ev.e_net / 1000), regen, fail

VARIANTS_22 = {
    "base":       dict(preview=False, energy_reward=False),
    "preview":    dict(preview=True,  energy_reward=False),
    "e-reward":   dict(preview=False, energy_reward=True),
    "preview+e":  dict(preview=True,  energy_reward=True),
}

nmpc_regen = 100. * ev_nmpc.e_recovered / max(ev_nmpc.e_propulsion, 1e-9)
nmpc_enet  = ev_nmpc.e_net / 1000

print("=" * 66)
print(f"STAGE A - 2x2 ABLATION (seed {PRIMARY_SEED}, "
      f"{int(np.sum(list(PHASE_STEPS.values()))):,} steps each)")
print(f"  Target to close: NMPC regen = {nmpc_regen:.1f}%  "
      f"(E_net = {nmpc_enet:.2f} kJ)")
print("=" * 66)

models_22, stats_22 = {}, {}
for name, fl in VARIANTS_22.items():
    print(f"\n> Variant '{name}'  (preview={fl['preview']}, "
          f"energy_reward={fl['energy_reward']})")
    models_22[name] = train_ppo_curriculum(PRIMARY_SEED, **fl)
    r, e, g, f = eval_policy_scurve(models_22[name], X0, **fl)
    stats_22[name] = (r, e, g, f)
    print(f"  -> RMSE_d={r:.3f} m | E_net={e:.2f} kJ | regen={g:.1f}%"
          + ("  (FAILED)" if f else ""))

print("\n" + "=" * 66)
print("GAP-CLOSING TABLE  (paper: main contribution evidence)")
print("=" * 66)
print(f"  {'Variant':<12} {'RMSE_d [m]':>11} {'E_net [kJ]':>11} "
      f"{'Regen %':>9} {'Gap closed':>11}")
print("  " + "-" * 60)
base_g = stats_22["base"][2]
for name in VARIANTS_22:
    r, e, g, f = stats_22[name]
    closed = 100. * (g - base_g) / max(nmpc_regen - base_g, 1e-9)
    print(f"  {name:<12} {r:>11.3f} {e:>11.2f} {g:>8.1f}% {closed:>10.0f}%"
          + ("  FAIL" if f else ""))
print(f"  {'NMPC (ref)':<12} {'-':>11} {nmpc_enet:>11.2f} "
      f"{nmpc_regen:>8.1f}% {'100':>10}%")
print("=" * 66)

# Winner selection
_ok = {k: v for k, v in stats_22.items() if v[0] <= 0.6 and not v[3]}
if not _ok:
    _ok = {k: v for k, v in stats_22.items() if not v[3]} or stats_22
WINNER = builtins.min(_ok, key=lambda k: _ok[k][1])
BEST_PREVIEW  = VARIANTS_22[WINNER]["preview"]
BEST_EREWARD  = VARIANTS_22[WINNER]["energy_reward"]
print(f"\nWINNER: '{WINNER}' (preview={BEST_PREVIEW}, "
      f"energy_reward={BEST_EREWARD}) - used for ALL downstream analyses.")

def make_env(dyn_m, vel_p, phase):
    """Factory: environment with the WINNER."""
    return PathFollowEnv(dyn_m, vel_p, phase=phase, t_step=T_SMPL,
                         preview=BEST_PREVIEW, energy_reward=BEST_EREWARD)

print("\n" + "=" * 66)
print(f"STAGE B - WINNER '{WINNER}' x {len(SEEDS)} SEEDS")
print("=" * 66)

seed_models, seed_stats = {}, {}
lc_callback = LearningCurveCallback(eval_freq=5000)
seed_models[PRIMARY_SEED] = models_22[WINNER]  # reuse Stage-A model
seed_stats[PRIMARY_SEED] = stats_22[WINNER][:2] + (stats_22[WINNER][3],)
print(f"  Seed {PRIMARY_SEED}: reused from Stage A "
      f"(RMSE_d={stats_22[WINNER][0]:.3f} m)")

for sd in SEEDS:
    if sd == PRIMARY_SEED:
        continue
    print(f"\n> Seed {sd}")
    seed_models[sd] = train_ppo_curriculum(
        sd, preview=BEST_PREVIEW, energy_reward=BEST_EREWARD)
    r, e, g, f = eval_policy_scurve(seed_models[sd], X0,
                                    preview=BEST_PREVIEW,
                                    energy_reward=BEST_EREWARD)
    seed_stats[sd] = (r, e, f)
    print(f"  Seed {sd}: RMSE_d={r:.3f} m | E_net={e:.2f} kJ | regen={g:.1f}%"
          + ("  (FAILED)" if f else ""))

rs = np.array([seed_stats[s][0] for s in SEEDS])
es = np.array([seed_stats[s][1] for s in SEEDS])
print("\n" + "=" * 66)
print(f"ACROSS-SEED SUMMARY  (winner '{WINNER}', N_seeds = {len(SEEDS)})")
print("=" * 66)
print(f"  RMSE_d : {np.nanmean(rs):.3f} +/- {np.nanstd(rs):.3f} m   "
      f"(min {np.nanmin(rs):.3f}, max {np.nanmax(rs):.3f})")
print(f"  E_net  : {np.nanmean(es):.2f} +/- {np.nanstd(es):.2f} kJ")
print(f"  Failures: {sum(int(seed_stats[s][2]) for s in SEEDS)}/{len(SEEDS)}")
print("=" * 66)

ppo_model = seed_models[PRIMARY_SEED]
ppo_model.save("ppo_pathfollow")
# keep straight path
models_22["base"].save("ppo_base_8d")
print(f"\nPrimary winner model saved: ppo_pathfollow.zip "
      f"(baseline: ppo_base_8d.zip)")

In [ ]:
# Policy evaluation and final comparison
try:
    ppo_eval = PPO.load("ppo_pathfollow", device="cpu")  # fair CPU timing
    print("Loaded saved model")
except:
    ppo_eval = ppo_model
    print("Using freshly trained model")

# Run simulation
eval_env   = make_env(dyn_model, vel_profile, 2)
obs_e, _   = eval_env.reset(seed=0)

# set benchmark IC
eval_env.state = dyn_model.clip_state(X0.copy())
s_c   = float(np.clip(X0[0], 0, path.s_max))
kappa = float(path.curvature(s_c))
v_ref = float(vel_profile.get_maximum_speed(s_c))
obs_e = eval_env._obs()

st_ppo, ct_ppo, tm_ppo = [eval_env.state.copy()], [], []
ev_ppo = EVEnergyModel(t_step=T_SMPL)

for _ in range(500):
    t0 = time.perf_counter()
    act, _ = ppo_eval.predict(obs_e, deterministic=True)
    ts = time.perf_counter() - t0
    obs_e, _, term, trunc, _ = eval_env.step(act)
    u = eval_env._denorm(act)
    st_ppo.append(eval_env.state.copy())
    ct_ppo.append(u); tm_ppo.append(ts)
    ev_ppo.step(float(eval_env.state[4]), float(u[1]), ts)
    if term or trunc or float(eval_env.state[0]) >= path.s_max - 0.1:
        break

st_ppo = np.array(st_ppo)
ct_ppo = np.array(ct_ppo) if ct_ppo else np.zeros((0,2))
tm_ppo = np.array(tm_ppo)
nmpc_avg_ms = times_nmpc.mean() * 1000

# Metrics 
if len(st_ppo) > 1:
    rmse_d_ppo = float(np.sqrt(np.mean(st_ppo[1:,1]**2)))
    speedup    = float(times_nmpc.mean() / tm_ppo.mean()) if tm_ppo.size > 0 else 0
    e_net_ppo  = float(ev_ppo.e_net / 1000)
    print(f"\nPPO | RMSE_d={rmse_d_ppo:.4f}m | Steps={len(tm_ppo)}")
    print(f"     Avg solve={tm_ppo.mean()*1000:.3f}ms | Speedup={speedup:.0f}x")
    print("Energy:"); ev_ppo.summary()
    print(f"     Regen fraction: {100.*ev_ppo.e_recovered/max(ev_ppo.e_propulsion,1e-9):.1f}% (NMPC: {100.*ev_nmpc.e_recovered/max(ev_nmpc.e_propulsion,1e-9):.1f}%)")
    print(f"     E_net={e_net_ppo:.3f} kJ")

    # Final comparison
    r_nmpc = float(np.sqrt(np.mean(states_nmpc[1:,1]**2)))
    r_pid  = float(np.sqrt(np.mean(states_pid[1:,1]**2)))
    print(f"\n{'='*70}")
    print(f"{'FINAL COMPARISON':^70}")
    print(f"{'='*70}")
    print(f"  {'Controller':<15} {'RMSE_d [m]':>12} {'Avg [ms]':>12} {'Speedup':>10} {'Steps':>8}")
    print(f"  {'-'*60}")
    e_nmpc = ev_nmpc.e_net / 1000
    e_pid  = ev_pid.e_net  / 1000
    print(f"  {'Controller':<15} {'RMSE_d [m]':>12} {'E_net [kJ]':>12} {'Avg [ms]':>10} {'Speedup':>10} {'Steps':>7}")
    print(f"  {'-'*70}")
    print(f"  {'NMPC (warm)':<15} {r_nmpc:>12.4f} {e_nmpc:>12.3f} {times_nmpc.mean()*1000:>10.1f} {'1x':>10} {len(times_nmpc):>7}")
    print(f"  {'PID (SF)':<15} {r_pid:>12.4f} {e_pid:>12.3f} {times_pid.mean()*1000:>10.3f} {times_nmpc.mean()/times_pid.mean():>9.0f}x {len(times_pid):>7}")
    if tm_ppo.size > 0:
        print(f"  {'PPO':<15} {rmse_d_ppo:>12.4f} {e_net_ppo:>12.3f} {tm_ppo.mean()*1000:>10.3f} {speedup:>9.0f}x {len(tm_ppo):>7}")
    print(f"{'='*70}")
    print(f"  Note: E_net = E_prop - E_regen + E_comp")
    print(f"  NMPC E_regen={ev_nmpc.e_recovered/1000:.2f}kJ (stopping profile)")

    # Plot
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    s_plt = np.linspace(0, path.s_max, 300)
    xy_ref  = path.xy(s_plt)
    xy_nmpc = np.array([path.sd_to_xy(s,d) for s,d in zip(states_nmpc[:,0],states_nmpc[:,1])])
    xy_pid  = np.array([path.sd_to_xy(s,d) for s,d in zip(states_pid[:,0], states_pid[:,1])])
    xy_ppo  = np.array([path.sd_to_xy(s,d) for s,d in zip(st_ppo[:,0],     st_ppo[:,1])])

    axes[0].plot(xy_ref[:,0],  xy_ref[:,1],  'k--', lw=2, label='Reference')
    axes[0].plot(xy_nmpc[:,0], xy_nmpc[:,1], 'r-',  lw=2, label='NMPC')
    axes[0].plot(xy_pid[:,0],  xy_pid[:,1],  'b-',  lw=2, label='PID')
    axes[0].plot(xy_ppo[:,0],  xy_ppo[:,1],  'g-',  lw=2, label='PPO')
    axes[0].set_aspect('equal'); axes[0].legend(); axes[0].grid(True)
    axes[0].set_title('Trajectory')

    t_n = np.arange(len(states_nmpc))*T_SMPL
    t_p = np.arange(len(st_ppo))*T_SMPL
    axes[1].plot(t_n, states_nmpc[:,1], 'r-', lw=2, label='NMPC')
    axes[1].plot(t_p, st_ppo[:,1],      'g-', lw=2, label='PPO')
    axes[1].axhline(0, color='k', ls='--')
    axes[1].legend(); axes[1].grid(True); axes[1].set_title('Lateral d [m]')

    axes[2].bar(['NMPC','PID','PPO'],
                [r_nmpc, r_pid, rmse_d_ppo],
                color=['red','blue','green'], alpha=0.75, edgecolor='k')
    axes[2].set_title('RMSE_d [m]'); axes[2].grid(True, axis='y')
    plt.suptitle('Controller Comparison', fontsize=13, fontweight='bold')
    plt.tight_layout(); plt.show()
else:
    print("PPO needs more training (FAST_TEST=False)")

In [ ]:
# Sensitivity to regeneration efficiency
ETA0 = 0.65
ETA_LEVELS = (0.50, 0.65, 0.80)

print("=" * 72)
print("ETA_R SENSITIVITY  (E_net = E_prop - (eta/0.65)*E_regen + E_cpu)")
print("=" * 72)
print(f"  {'Controller':<10} " + "".join(f"{'eta='+str(e):>14}" for e in ETA_LEVELS)
      + f" {'Regen frac @0.65':>18}")
print("  " + "-" * 68)
for name, ev in [("NMPC", ev_nmpc), ("PPO", ev_ppo),
                 ("PID-SF", ev_pid), ("Stanley", ev_stan)]:
    row = []
    for eta in ETA_LEVELS:
        e_net = (ev.e_propulsion - (eta / ETA0) * ev.e_recovered
                 + ev.e_comp) / 1000
        row.append(f"{e_net:>12.2f}kJ")
    frac = ev.e_recovered / max(ev.e_propulsion, 1e-9) * 100
    print(f"  {name:<10} " + "".join(row) + f" {frac:>16.1f}%")
print("=" * 72)
print("Note: regen fraction E_r/E_p is eta-dependent through E_r; the")
print("controller RANKING by E_net is what matters for the paper claim.")

In [ ]:
# Hybrid baseline: PPO steering + analytic stop 
L_STOP = V_MAX ** 2 / (2.0 * A_LNG_MAX) + 1.5  # ~12.2
KP_STOP = 1.5  # P-gain on (v_ref)

env_h = make_env(dyn_model, vel_profile, 2)
env_h.reset(seed=0)
env_h.state = dyn_model.clip_state(X0.copy())
obs_h = env_h._obs()
st_h = [env_h.state.copy()]
ev_h = EVEnergyModel(t_step=T_SMPL)
n_override = 0

for _ in range(500):
    act_h, _ = ppo_eval.predict(obs_h, deterministic=True)
    act_h = np.array(act_h, dtype=np.float32).copy()
    s_now = float(env_h.state[0])
    if s_now >= path.s_max - L_STOP:
        v_now = float(env_h.state[4])
        v_ref_now = float(vel_profile.get_maximum_speed(
            float(np.clip(s_now, 0., path.s_max))))
        _u2m = float(dyn_model.control_upper[1])
        u2_cmd = float(np.clip(KP_STOP * (v_ref_now - v_now), -_u2m, _u2m))
        act_h[1] = u2_cmd / _u2m  # renormalize for env.step
        n_override += 1
    obs_h, _, term_h, trunc_h, _ = env_h.step(act_h)
    u_h = env_h._denorm(act_h)
    ev_h.step(float(env_h.state[4]), float(u_h[1]), 0.)
    st_h.append(env_h.state.copy())
    if term_h or trunc_h or float(env_h.state[0]) >= path.s_max - 0.1:
        break

st_h = np.array(st_h)
rmse_h = float(np.sqrt(np.mean(st_h[1:, 1] ** 2)))
regen_h = 100. * ev_h.e_recovered / max(ev_h.e_propulsion, 1e-9)
nmpc_regen_h = 100. * ev_nmpc.e_recovered / max(ev_nmpc.e_propulsion, 1e-9)

print("=" * 64)
print("HYBRID (PPO steering + analytic stop)  vs  pure controllers")
print("=" * 64)
print(f"  {'Controller':<22} {'RMSE_d [m]':>11} {'E_net [kJ]':>11} {'Regen %':>9}")
print("  " + "-" * 58)
print(f"  {'PPO (winner, pure)':<22} {rmse_d_ppo:>11.3f} "
      f"{ev_ppo.e_net/1000:>11.2f} "
      f"{100.*ev_ppo.e_recovered/max(ev_ppo.e_propulsion,1e-9):>8.1f}%")
print(f"  {'Hybrid PPO+stop':<22} {rmse_h:>11.3f} {ev_h.e_net/1000:>11.2f} "
      f"{regen_h:>8.1f}%")
print(f"  {'NMPC (ref)':<22} "
      f"{float(np.sqrt(np.mean(states_nmpc[1:,1]**2))):>11.3f} "
      f"{ev_nmpc.e_net/1000:>11.2f} {nmpc_regen_h:>8.1f}%")
print("=" * 64)
print(f"  Longitudinal override active on {n_override} steps "
      f"(last {L_STOP:.1f} m).")
print("  Paper: report as one row; the delta (Hybrid - PPO) isolates the")
print("  stopping-maneuver share of the energy gap.")

In [ ]:
# NMPC weight Pareto analysis
from scipy.stats import qmc

# Sampling config
N_SAMPLES = 30  # 30 x ~15s/sim

# Weight bounds
WEIGHT_BOUNDS = {
    "d"    : (0.0, 2.0),  # 1 100
    "alpha": (0.0, 3.0),  # 1 1000
    "v"    : (-1.0, 1.0),  # 0.1 10
    "u1"   : (-1.0, 1.0),
    "u2"   : (-1.0, 1.0),
}

def lhs_sample_weights(n, bounds, seed=42):
    keys    = list(bounds.keys())
    lo      = np.array([bounds[k][0] for k in keys])
    hi      = np.array([bounds[k][1] for k in keys])
    sampler = qmc.LatinHypercube(d=len(keys), seed=seed)
    unit    = sampler.random(n=n)
    scaled  = qmc.scale(unit, lo, hi)
    return [dict(zip(keys, 10.0**row)) for row in scaled]

def is_pareto_optimal(costs):
    """Return boolean mask - True."""
    n = len(costs); mask = np.ones(n, dtype=bool)
    for i in range(n):
        if not mask[i]: continue
        for j in range(n):
            if i==j or not mask[j]: continue
            if np.all(costs[j]<=costs[i]) and np.any(costs[j]<costs[i]):
                mask[i]=False; break
    return mask

weight_samples = lhs_sample_weights(N_SAMPLES, WEIGHT_BOUNDS)
print(f"Running {N_SAMPLES} simulations...")

pareto_results = []
for idx, w in enumerate(weight_samples):
    try:
        _mpc = NMPCModel(dyn_model=dyn_model, vel_profile=vel_profile,
                         obj_weights=dict(w), L_f=L_F, t_step=T_SMPL,
                         n_horizon=N_HORIZON)
        _ev  = EVEnergyModel(t_step=T_SMPL)
        _st, _, _, _ = _mpc.simulate(x0=X0.copy(), ev_energy_model=_ev,
                                   warm_start=True, nr_steps=200, file_name=None)
        rmse_d = float(np.sqrt(np.mean(_st[1:,1]**2))) if len(_st)>1 else float("nan")
        e_net  = float(_ev.e_net/1000)
    except:
        rmse_d, e_net = float("nan"), float("nan")
    pareto_results.append({**w, "rmse_d": rmse_d, "e_net": e_net})
    if (idx+1) % 5 == 0:
        print(f"  {idx+1}/{N_SAMPLES} done")

# Pareto analysis
valid = [(r,i) for i,r in enumerate(pareto_results)
         if np.isfinite(r["rmse_d"]) and np.isfinite(r["e_net"])]
costs = np.array([[r["rmse_d"],r["e_net"]] for r,_ in valid])
mask  = is_pareto_optimal(costs)

pareto_mask = np.zeros(len(pareto_results), dtype=bool)
for flag, (_,oi) in zip(mask, valid): pareto_mask[oi] = flag

# Plot
rmse_all = np.array([r["rmse_d"] for r in pareto_results])
enet_all = np.array([r["e_net"]  for r in pareto_results])
v_ok     = np.isfinite(rmse_all) & np.isfinite(enet_all)

fig, ax = plt.subplots(figsize=(9,6))
ax.scatter(rmse_all[v_ok & ~pareto_mask], enet_all[v_ok & ~pareto_mask],
           c="steelblue", alpha=0.5, s=40, label="All samples")
ax.scatter(rmse_all[v_ok & pareto_mask],  enet_all[v_ok & pareto_mask],
           c="crimson", s=80, zorder=5, label="Pareto front")

pf_i = np.where(v_ok & pareto_mask)[0]
pf_s = pf_i[np.argsort(rmse_all[pf_i])]
ax.plot(rmse_all[pf_s], enet_all[pf_s], "r--", lw=1.2, alpha=0.7)
ax.set_xlabel("RMSE_d [m]  (tracking - lower is better)", fontsize=12)
ax.set_ylabel("E_net [kJ]  (energy - lower is better)",   fontsize=12)
ax.set_title(f"Pareto Front - NMPC Weights (N={N_SAMPLES})",
             fontsize=13, fontweight="bold")
ax.legend(fontsize=11); ax.grid(True, alpha=0.4)
plt.tight_layout(); plt.show()

# Print optimal
pf_res = [r for r,i in zip(pareto_results, range(len(pareto_results))) if pareto_mask[i]]
pf_res = sorted(pf_res, key=lambda r: r["rmse_d"])
print(f"\n{'='*65}")
print(f"PARETO-OPTIMAL WEIGHT SETS ({len(pf_res)} found)")
print(f"{'='*65}")
print(f"{'d':>8} {'alpha':>8} {'v':>6} {'u1':>6} {'u2':>6}  {'RMSE_d':>8} {'E_net':>8}")
print("-"*65)
for r in pf_res:
    print(f"{r['d']:8.2f} {r['alpha']:8.2f} {r['v']:6.2f} {r['u1']:6.2f} {r['u2']:6.2f}  "
          f"{r['rmse_d']:8.4f} {r['e_net']:8.3f}")

In [ ]:
# Dynamic bicycle replay validation
_M, _L_F, _L_R = 1500.0, 2.0, 2.0
_I_ZZ = 2500.0
_C_F = _C_R = 80000.0  # N/rad linear Pacejka
_V_MIN = 0.5  # m/s matches kinematic

def _dyn_step(st, delta, ax, dt=0.1):
    """RK4 step: dynamic bicycle model."""
    def f(s):
        _, _, ps, vx, vy, r = s
        vx = max(vx, _V_MIN)
        af = delta - (vy + _L_F * r) / vx
        ar =       - (vy - _L_R * r) / vx
        Ff, Fr = _C_F * af, _C_R * ar
        return np.array([
            vx*np.cos(ps) - vy*np.sin(ps),
            vx*np.sin(ps) + vy*np.cos(ps),
            r,
            ax + r*vy,
            (Ff + Fr)/_M - r*vx,
            (_L_F*Ff - _L_R*Fr)/_I_ZZ,
        ])
    k1 = f(st); k2 = f(st+0.05*k1); k3 = f(st+0.05*k2); k4 = f(st+dt*k3)
    return st + dt*(k1 + 2*k2 + 2*k3 + k4)/6

def _replay_dynamic(controls, x0_global, delta0=0.0):
    """Replay kinematic control sequence on."""
    st, delta_k = np.array(x0_global, float), float(delta0)
    d_list = []
    for u in controls:
        delta_k = np.clip(delta_k + float(u[0]) * 0.1, -0.6, 0.6)
        st = _dyn_step(st, delta_k, float(u[1]))
        try:
            _, d_est = path.xy_to_sd(float(st[0]), float(st[1]))
        except Exception:
            d_est = float('nan')
        d_list.append(d_est)
    return np.array(d_list)

# Initial condition
X0_DYN = np.array([0.0, 0.5, 0.1, 0.0, 8.0])
_x0g, _y0g = path.sd_to_xy(X0_DYN[0], X0_DYN[1])
_tang0 = path.tangent(X0_DYN[0])
_psi0  = np.arctan2(_tang0[1], _tang0[0]) + X0_DYN[2]
_x0_global = np.array([_x0g, _y0g, _psi0,
                        X0_DYN[4]*np.cos(X0_DYN[2]),
                        X0_DYN[4]*np.sin(X0_DYN[2]), 0.0])

print("=" * 65)
print("DYNAMIC BICYCLE MODEL VALIDATION  (v0 = 8 m/s)")
print("  Cf = Cr = 80 kN/rad  (linear Pacejka)")
print("  Control integration: delta_dot -> delta (rate->angle)")
print("=" * 65)

# NMPC
_d_kin_n = states_nmpc[1:, 1]
_d_dyn_n = _replay_dynamic(controls_nmpc, _x0_global, delta0=X0_DYN[3])
_n_n = min(len(_d_kin_n), len(_d_dyn_n))
_rk_n = float(np.sqrt(np.nanmean(_d_kin_n[:_n_n]**2)))
_rd_n = float(np.sqrt(np.nanmean(_d_dyn_n[:_n_n]**2)))
_stab_n = bool(np.all(np.abs(_d_dyn_n[:_n_n]) < 4.9)) if _n_n > 0 else False
_dg_n = (_rd_n - _rk_n) / _rk_n * 100 if _rk_n > 0 else float('nan')

print()
print(f"NMPC  kinematic  RMSE_d = {_rk_n:.3f} m")
print(f"NMPC  dynamic    RMSE_d = {_rd_n:.3f} m  ({_dg_n:+.1f}%)")
print(f"      Stability : " + ("stable (|d| < 4.9 m)" if _stab_n else "unstable"))

# PPO
_env_d = make_env(dyn_model, vel_profile, 2)
_env_d.state = dyn_model.clip_state(X0_DYN.copy())
_obs_d = _env_d._obs()
_ctrl_p, _d_kin_p = [], []

for _ in range(500):
    _act, _ = ppo_eval.predict(_obs_d, deterministic=True)
    _obs_d, _, _t, _tr, _ = _env_d.step(_act)
    _ctrl_p.append(_env_d._denorm(_act).copy())
    _d_kin_p.append(float(_env_d.state[1]))
    if _t or _tr or float(_env_d.state[0]) >= path.s_max - 0.1:
        break

_ctrl_arr  = np.array(_ctrl_p)
_d_kin_arr = np.array(_d_kin_p)
_d_dyn_p   = _replay_dynamic(_ctrl_arr, _x0_global, delta0=X0_DYN[3])
_n_p = min(len(_d_kin_arr), len(_d_dyn_p))
_rk_p = float(np.sqrt(np.nanmean(_d_kin_arr[:_n_p]**2)))
_rd_p = float(np.sqrt(np.nanmean(_d_dyn_p[:_n_p]**2)))
_stab_p = bool(np.all(np.abs(_d_dyn_p[:_n_p]) < 4.9)) if _n_p > 0 else False
_dg_p = (_rd_p - _rk_p) / _rk_p * 100 if _rk_p > 0 else float('nan')

print()
print(f"PPO   kinematic  RMSE_d = {_rk_p:.3f} m")
print(f"PPO   dynamic    RMSE_d = {_rd_p:.3f} m  ({_dg_p:+.1f}%)")
print(f"      Stability : " + ("stable (|d| < 4.9 m)" if _stab_p else "unstable"))

print()
print("=" * 65)
print("SUMMARY")
print("=" * 65)
print(f"  {'Controller':<12} {'Kinematic':>10} {'Dynamic':>10} {'Delta':>10}")
print("  " + "-"*46)
for _nm, _rk, _rd, _dg in [("NMPC", _rk_n, _rd_n, _dg_n),
                             ("PPO",  _rk_p, _rd_p, _dg_p)]:
    if not (np.isnan(_rk) or np.isnan(_rd)):
        print(f"  {_nm:<12} {_rk:>10.3f} m {_rd:>10.3f} m {_dg:>+9.1f}%")
print("  " + "-"*46)
print("Note: v0 = 8 m/s worst-case, linear Pacejka (Cf=Cr=80 kN/rad).")
print("Nonlinear Pacejka validation deferred to future hardware work.")

In [ ]:
# Cross-paradigm Pareto: PPO energy-weight sweep
W_E_SWEEP = [0.0, 0.15, 0.30, 0.60]
DEFAULT_WE = PathFollowEnv._WEIGHTS[4]["w_energy"]  # 0.30

ppo_sweep = {}
for we in W_E_SWEEP:
    if abs(we - DEFAULT_WE) < 1e-9:
        # winner primary model
        print(f"w_e={we:.2f}: reusing winner primary model")
        mdl = seed_models[PRIMARY_SEED]
    else:
        print(f"w_e={we:.2f}: training (winner arch, seed {PRIMARY_SEED})")
        mdl = train_ppo_curriculum(PRIMARY_SEED,
                                   preview=BEST_PREVIEW,
                                   energy_reward=BEST_EREWARD,
                                   w_energy_ph4=we)
    r, e, g, f = eval_policy_scurve(mdl, X0, preview=BEST_PREVIEW,
                                    energy_reward=BEST_EREWARD)
    ppo_sweep[we] = (r, e, g, f)
    print(f"  w_e={we:.2f}: RMSE_d={r:.3f} m | E_net={e:.2f} kJ | "
          f"regen={g:.1f}%" + ("  FAIL" if f else ""))

# Overlay plot
fig, ax = plt.subplots(figsize=(9, 6))
# NMPC LHS cloud
ax.scatter(rmse_all[v_ok & ~pareto_mask], enet_all[v_ok & ~pareto_mask],
           c="steelblue", alpha=0.35, s=35, label="NMPC LHS samples")
pf_i = np.where(v_ok & pareto_mask)[0]
pf_s = pf_i[np.argsort(rmse_all[pf_i])]
ax.plot(rmse_all[pf_s], enet_all[pf_s], "b--o", lw=1.4, ms=6,
        label="NMPC Pareto front")
# NMPC default
ax.scatter([float(np.sqrt(np.mean(states_nmpc[1:, 1]**2)))],
           [ev_nmpc.e_net / 1000], marker="*", s=260, c="gold",
           edgecolors="k", zorder=6, label="NMPC default")
# PPO sweep
_wes = sorted(ppo_sweep.keys())
_rr = [ppo_sweep[w][0] for w in _wes]
_ee = [ppo_sweep[w][1] for w in _wes]
ax.plot(_rr, _ee, "g-^", lw=1.6, ms=8, zorder=5,
        label=f"PPO w_e sweep ({WINNER})")
for w, r, e in zip(_wes, _rr, _ee):
    ax.annotate(f"w_e={w:g}", (r, e), textcoords="offset points",
                xytext=(6, 5), fontsize=9, color="darkgreen")
# classical fixed points
ax.scatter([float(np.sqrt(np.mean(states_pid[1:, 1]**2)))],
           [ev_pid.e_net / 1000], marker="s", s=90, c="crimson",
           edgecolors="k", zorder=5, label="PID-SF")
ax.scatter([float(np.sqrt(np.mean(states_stan[1:, 1]**2)))],
           [ev_stan.e_net / 1000], marker="D", s=80, c="darkorange",
           edgecolors="k", zorder=5, label="Stanley")

ax.set_xlabel("RMSE_d [m]  (tracking - lower is better)", fontsize=12)
ax.set_ylabel("E_net [kJ]  (energy - lower is better)", fontsize=12)
ax.set_title("Cross-Paradigm Tracking-Energy Pareto Comparison",
             fontsize=13, fontweight="bold")
ax.legend(fontsize=10); ax.grid(True, alpha=0.4)
plt.tight_layout()
plt.savefig("fig_cross_pareto.pdf", bbox_inches="tight")
plt.show()
print("fig_cross_pareto.pdf saved")

In [ ]:
# Learning curves
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

if len(lc_callback.steps) > 0:
    fig, ax = plt.subplots(figsize=(8, 3.5))
    ax.plot(lc_callback.steps, lc_callback.rewards,
            color='#1a3a5c', lw=1.5, alpha=0.8)

    # Shade each phase
    phase_colors = ['#d0e8f8','#e8f4f8','#d5e8d4','#fff2cc','#f8cecc']
    phase_labels = ['Ph.0\nStraight', 'Ph.1\nS-curve easy',
                    'Ph.2\nStopping', 'Ph.3\nDisturbance',
                    'Ph.4\nEnergy']
    cumsteps = [0]
    for n in [PHASE_STEPS[p] for p in (0,1,2,3,4)]:
        cumsteps.append(cumsteps[-1] + n)

    for i in range(30):
        ax.axvspan(cumsteps[i], cumsteps[i+1],
                   alpha=0.25, color=phase_colors[i],
                   label=phase_labels[i])
        ax.axvline(cumsteps[i+1], color='gray', lw=0.8, ls='--')

    ax.set_xlabel('Training Steps')
    ax.set_ylabel('Mean Episode Reward')
    ax.set_title('PPO Learning Curve - 5-Phase Curriculum')
    ax.legend(loc='lower right', fontsize=8, ncol=4)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('fig_learning_curve.pdf', dpi=150, bbox_inches='tight')
    plt.show()
    print("fig_learning_curve.pdf saved ")
else:
    print("No learning curve data (FAST_TEST may have been used)")

In [ ]:
# Phase portrait
fig, axes = plt.subplots(1, 3, figsize=(12, 3.8))
fig.subplots_adjust(wspace=0.35)

entries = [
    (states_nmpc, 'NMPC',   '#1a3a5c'),
    (st_ppo,      'PPO',    '#2e6da4'),
    (states_pid,  'PID-SF', '#1a1a1a'),
]

for ax, (st, name, col) in zip(axes, entries):
    ax.plot(st[:,1], st[:,2], color=col, lw=1.5, alpha=0.85)
    ax.scatter(st[0,1],  st[0,2],  marker='o', s=80,
               color=col, zorder=5, label='Start')
    ax.scatter(st[-1,1], st[-1,2], marker='*', s=120,
               color='red', zorder=5, label='End')
    ax.axhline(0, color='k', lw=0.7, ls=':')
    ax.axvline(0, color='k', lw=0.7, ls=':')
    ax.set_xlabel('Lateral deviation d (m)')
    ax.set_ylabel('Heading error alpha (rad)')
    ax.set_title(f'{name}')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
    # Annotate RMSE
    rm = float(np.sqrt(np.mean(st[1:,1]**2)))
    ax.text(0.97, 0.97, f'RMSE$_d$={rm:.3f}m',
            transform=ax.transAxes, ha='right', va='top',
            fontsize=8, bbox=dict(boxstyle='round',
            facecolor='white', edgecolor='#aaaaaa', alpha=0.8))

fig.suptitle('Phase Portrait: Lateral State Space (d, alpha)',
             fontsize=10, fontweight='bold')
plt.savefig('fig_phase_portrait.pdf', dpi=150, bbox_inches='tight')
plt.show()
print("fig_phase_portrait.pdf saved ")

In [ ]:
# PID-SF step response
# Use straight path
dyn_step = FrenetDynamicModel(
    path=path_straight,
    state_bounds=np.array([[0,path_straight.s_max],
        [-D_MAX,D_MAX],[-ALPHA_MAX,ALPHA_MAX],
        [-DELTA_MAX,DELTA_MAX],[0,V_MAX]]),
    control_bounds=control_bounds, L_f=L_F, t_step=T_SMPL)
vel_step_p = ReferenceVelocityGenerator(
    path_straight, max_speed=5.0,
    max_lateral_acceleration=A_LAT_MAX,
    max_deceleration=A_LNG_MAX,
    nr_path_samples=100, filter_window=3)
pid_step = PIDModel(dyn_model=dyn_step, vel_profile=vel_step_p,
                    L_f=L_F, t_step=T_SMPL,
                    Kff_kappa=0.0,  # no feedforward
                    Kp_lon=1.0, Ki_lon=0.02)
ev_step = EVEnergyModel(t_step=T_SMPL)

X0_step = np.array([0.0, 1.0, 0.0, 0.0, 5.0])  # d0=1m on straight
st_step, _, _, _ = pid_step.simulate(
    x0=X0_step.copy(), ev_energy_model=ev_step, nr_steps=200)

t_step_vec = np.arange(len(st_step)) * T_SMPL
d_step     = st_step[:,1]

# Find settling time
SETTLE_TOL = 0.05
settled_idx = None
for i in range(len(d_step)-10):
    if np.all(np.abs(d_step[i:i+10]) < SETTLE_TOL):
        settled_idx = i; break

t_settle = t_step_vec[settled_idx] if settled_idx else float('nan')

print(f"PID-SF Step Response (d0=1m, v=5m/s):")
print(f"  Settling time (|d|<0.05m): {t_settle:.2f} s")
print(f"  Final deviation:            {d_step[-1]:.4f} m")
print(f"  Overshoot:                  {max(0, d_step.min()):.4f} m")

fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(t_step_vec, d_step, color='#1a1a1a', lw=1.8)
ax.axhline(0, color='k', lw=0.7, ls=':')
ax.axhline(SETTLE_TOL, color='r', lw=0.8, ls='--', alpha=0.6, label=f'+/-{SETTLE_TOL}m band')
ax.axhline(-SETTLE_TOL, color='r', lw=0.8, ls='--', alpha=0.6)
if settled_idx:
    ax.axvline(t_settle, color='g', lw=1.2, ls='--',
               label=f'$t_s$={t_settle:.2f}s')
ax.set_xlabel('Time (s)'); ax.set_ylabel('d (m)')
ax.set_title('PID-SF Step Response (d0=1m, v=5m/s)')
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('fig_pid_step.pdf', dpi=150, bbox_inches='tight')
plt.show()
print("fig_pid_step.pdf saved ")

In [ ]:
# Statistical validation over N=30 LHS conditions
from scipy.stats import qmc as _qmc

N_STAT  = 30
sampler = _qmc.LatinHypercube(d=3, seed=42)
lhs_u   = sampler.random(n=N_STAT)
lhs_sc  = _qmc.scale(lhs_u,
                      l_bounds=[-1.5, -0.2, 2.0],
                      u_bounds=[ 1.5,  0.2, 6.0])
ICS_30  = [np.array([0.0, row[0], row[1], 0.0, row[2]])
           for row in lhs_sc]
print(f"Generated {N_STAT} LHS initial conditions")

res30 = {c: {"rmse_d": [], "e_net": [], "t_ms": [], "fail": []}
         for c in ["NMPC", "PPO", "PID", "STAN"]}

for ic_i, x0_ic in enumerate(ICS_30):
    if (ic_i + 1) % 5 == 0:
        print(f"  IC {ic_i+1}/{N_STAT}  d0={x0_ic[1]:.2f}m  v0={x0_ic[4]:.1f}m/s")

    # NMPC
    try:
        _m = NMPCModel(dyn_model=dyn_model, vel_profile=vel_profile,
                       obj_weights=dict(obj_weights), L_f=L_F,
                       t_step=T_SMPL, n_horizon=N_HORIZON)
        _ev = EVEnergyModel(t_step=T_SMPL)
        _st, _, _tm, _fl = _m.simulate(x0=x0_ic.copy(),
                                       ev_energy_model=_ev,
                                       warm_start=True, nr_steps=200,
                                       file_name=None)
        res30["NMPC"]["rmse_d"].append(float(np.sqrt(np.mean(_st[1:, 1] ** 2))))
        res30["NMPC"]["e_net"].append(float(_ev.e_net / 1000))
        res30["NMPC"]["t_ms"].append(float(_tm.mean() * 1000))
        res30["NMPC"]["fail"].append(bool(_fl))
    except Exception:
        for k in ("rmse_d", "e_net", "t_ms"):
            res30["NMPC"][k].append(float("nan"))
        res30["NMPC"]["fail"].append(True)

    # PPO
    try:
        _env = make_env(dyn_model, vel_profile, 2)
        _env.reset(seed=1000 + ic_i)
        _env.state = dyn_model.clip_state(x0_ic.copy())
        _obs = _env._obs()
        _st_r, _tm_r, _ev_r = [], [], EVEnergyModel(t_step=T_SMPL)
        _fail_r = False
        for _ in range(500):
            _t0 = time.perf_counter()
            _act, _ = ppo_eval.predict(_obs, deterministic=True)
            _ts = time.perf_counter() - _t0
            _obs, _, _term, _trunc, _info = _env.step(_act)
            _u = _env._denorm(_act)
            _ev_r.step(float(_env.state[4]), float(_u[1]), _ts)
            _st_r.append(_env.state.copy()); _tm_r.append(_ts)
            _fail_r = _fail_r or bool(_info.get("failed", False))
            if _term or _trunc or float(_env.state[0]) >= path.s_max - 0.1:
                break
        _arr = np.array(_st_r)
        res30["PPO"]["rmse_d"].append(float(np.sqrt(np.mean(_arr[:, 1] ** 2))))
        res30["PPO"]["e_net"].append(float(_ev_r.e_net / 1000))
        res30["PPO"]["t_ms"].append(float(np.mean(_tm_r) * 1000))
        res30["PPO"]["fail"].append(_fail_r)
    except Exception:
        for k in ("rmse_d", "e_net", "t_ms"):
            res30["PPO"][k].append(float("nan"))
        res30["PPO"]["fail"].append(True)

    # PID
    try:
        _pid2 = PIDModel(dyn_model=dyn_model, vel_profile=vel_profile,
                         L_f=L_F, t_step=T_SMPL, Kff_kappa=1.0,
                         Kp_lon=1.0, Ki_lon=0.02)
        _ev2 = EVEnergyModel(t_step=T_SMPL)
        _st2, _, _tm2, _fl2 = _pid2.simulate(x0=x0_ic.copy(),
                                             ev_energy_model=_ev2,
                                             nr_steps=200)
        res30["PID"]["rmse_d"].append(float(np.sqrt(np.mean(_st2[1:, 1] ** 2))))
        res30["PID"]["e_net"].append(float(_ev2.e_net / 1000))
        res30["PID"]["t_ms"].append(float(_tm2.mean() * 1000))
        res30["PID"]["fail"].append(bool(_fl2))
    except Exception:
        for k in ("rmse_d", "e_net", "t_ms"):
            res30["PID"][k].append(float("nan"))
        res30["PID"]["fail"].append(True)

    # Stanley
    try:
        _stan2 = StanleyModel(dyn_model=dyn_model, vel_profile=vel_profile,
                              L_f=L_F, t_step=T_SMPL,
                              k_e=1.0, k_soft=1.0, K_delta=5.0,
                              Kff_kappa=1.0, Kp_lon=1.0, Ki_lon=0.02)
        _ev3 = EVEnergyModel(t_step=T_SMPL)
        _st3, _, _tm3, _fl3 = _stan2.simulate(x0=x0_ic.copy(),
                                              ev_energy_model=_ev3,
                                              nr_steps=200)
        res30["STAN"]["rmse_d"].append(float(np.sqrt(np.mean(_st3[1:, 1] ** 2))))
        res30["STAN"]["e_net"].append(float(_ev3.e_net / 1000))
        res30["STAN"]["t_ms"].append(float(_tm3.mean() * 1000))
        res30["STAN"]["fail"].append(bool(_fl3))
    except Exception:
        for k in ("rmse_d", "e_net", "t_ms"):
            res30["STAN"][k].append(float("nan"))
        res30["STAN"]["fail"].append(True)

t_nmpc_stat = float(np.nanmean(res30["NMPC"]["t_ms"]))
print("\n" + "=" * 75)
print(f"{'STATISTICAL RESULTS  N=30 LHS ICs':^75}")
print("=" * 75)
print(f"  {'Controller':<10} {'RMSE_d [m]':^20} {'E_net [kJ]':^20} "
      f"{'Speedup':>9} {'Fail':>6}")
print("  " + "-" * 71)
for ctrl in ["NMPC", "PPO", "PID", "STAN"]:
    r = np.array(res30[ctrl]["rmse_d"]); e = np.array(res30[ctrl]["e_net"])
    t = np.array(res30[ctrl]["t_ms"]);   f = np.array(res30[ctrl]["fail"])
    rm, rs = float(np.nanmean(r)), float(np.nanstd(r))
    em, es = float(np.nanmean(e)), float(np.nanstd(e))
    sp = t_nmpc_stat / float(np.nanmean(t)) if ctrl != "NMPC" else 1.0
    sp_str = f"{sp:.0f}x" if ctrl != "NMPC" else "1x"
    print(f"  {ctrl:<10} {rm:.3f} +/- {rs:.3f}      {em:.2f} +/- {es:.2f}      "f"{sp_str:>9} {int(f.sum()):>3}/{N_STAT}")
print("=" * 75)
print("Note: failure = off-road (|d| >= d_max - 0.1) before end of path.")

In [ ]:
# Significance tests and effect sizes 
from scipy.stats import wilcoxon, mannwhitneyu

r_nmpc_all = np.array(res30["NMPC"]["rmse_d"])
r_ppo_all  = np.array(res30["PPO"]["rmse_d"])
r_pid_all  = np.array(res30["PID"]["rmse_d"])

# pairwise-finite mask
mask_np = np.isfinite(r_nmpc_all) & np.isfinite(r_ppo_all)
mask_pp = np.isfinite(r_pid_all)  & np.isfinite(r_ppo_all)

r_nmpc = r_nmpc_all[mask_np]
r_ppo  = r_ppo_all[mask_np]

# Paired test
_, p_wx = wilcoxon(r_ppo, r_nmpc, alternative='less')
# Unpaired test
_, p_mw = mannwhitneyu(r_ppo, r_nmpc, alternative='less')

def cohens_d(a, b):
    pooled = np.sqrt((np.std(a, ddof=1) ** 2 + np.std(b, ddof=1) ** 2) / 2)
    return (np.mean(b) - np.mean(a)) / pooled if pooled > 0 else 0.0

d_ppo_nmpc = cohens_d(r_ppo, r_nmpc)
d_ppo_pid  = cohens_d(r_ppo_all[mask_pp], r_pid_all[mask_pp])

def effect_label(d):
    ad = builtins.abs(d)
    return 'large' if ad > 0.8 else ('medium' if ad > 0.5 else 'small')

print("\n" + "=" * 62)
print("STATISTICAL TESTS  (N pairs = %d)" % len(r_ppo))
print("=" * 62)
print(f"  Wilcoxon signed-rank (paired, PPO < NMPC): p={p_wx:.4f}  "
      f"{'SIGNIFICANT' if p_wx < 0.05 else 'n.s.'}")
print(f"  Mann-Whitney U (unpaired,   PPO < NMPC): p={p_mw:.4f}")
print(f"  Cohen's d (PPO vs NMPC): d={d_ppo_nmpc:.3f} ({effect_label(d_ppo_nmpc)})")
print(f"  Cohen's d (PPO vs PID) : d={d_ppo_pid:.3f} ({effect_label(d_ppo_pid)})")
print("=" * 62)
print("\nFor paper:")
print(f"  Wilcoxon (paired) p={p_wx:.4f}; Mann-Whitney p={p_mw:.4f}; "
      f"Cohen's d={d_ppo_nmpc:.3f} ({effect_label(d_ppo_nmpc)})")

In [ ]:
# Zero-shot evaluation on unseen tracks 
state_bounds_lc = np.array([
    [0, path_lc.s_max],
    [-D_MAX, D_MAX], [-ALPHA_MAX, ALPHA_MAX],
    [-DELTA_MAX, DELTA_MAX], [0, V_MAX],
])
dyn_lc = FrenetDynamicModel(
    path=path_lc, state_bounds=state_bounds_lc,
    control_bounds=control_bounds, L_f=L_F, t_step=T_SMPL)
vel_lc = ReferenceVelocityGenerator(
    path_lc, max_speed=V_MAX,
    max_lateral_acceleration=A_LAT_MAX,
    max_deceleration=A_LNG_MAX,
    nr_path_samples=100, filter_window=3)
print(f"Lane-change model ready | s_max={path_lc.s_max:.2f}m")

state_bounds_ch = np.array([
    [0, path_chicane.s_max],
    [-D_MAX, D_MAX], [-ALPHA_MAX, ALPHA_MAX],
    [-DELTA_MAX, DELTA_MAX], [0, V_MAX]])
dyn_ch = FrenetDynamicModel(path=path_chicane,
    state_bounds=state_bounds_ch,
    control_bounds=control_bounds, L_f=L_F, t_step=T_SMPL)
vel_ch = ReferenceVelocityGenerator(path_chicane,
    max_speed=V_MAX, max_lateral_acceleration=A_LAT_MAX,
    max_deceleration=A_LNG_MAX,
    nr_path_samples=100, filter_window=3)
print(f"Chicane model ready    | s_max={path_chicane.s_max:.2f}m")

def _run_ppo_track(dyn_t, vel_t, path_t, x0_t, max_steps=800):
    """Zero-shot PPO rollout on an."""
    _env_t = make_env(dyn_t, vel_t, 2)
    _env_t.reset(seed=0)
    _env_t.state = dyn_t.clip_state(x0_t.copy())
    _obs_t = _env_t._obs()
    st_t = [_env_t.state.copy()]
    ev_t = EVEnergyModel(t_step=T_SMPL)
    fail_t = False
    for _ in range(max_steps):
        _act_t, _ = ppo_eval.predict(_obs_t, deterministic=True)
        _obs_t, _, _tm, _tr, _inf = _env_t.step(_act_t)
        _u_t = _env_t._denorm(_act_t)
        ev_t.step(float(_env_t.state[4]), float(_u_t[1]), 0.)
        st_t.append(_env_t.state.copy())
        fail_t = fail_t or bool(_inf.get("failed", False))
        if _tm or _tr or float(_env_t.state[0]) >= path_t.s_max - 0.1:
            break
    return np.array(st_t), ev_t, fail_t

X0_LC = np.array([0.0, 0.3, 0.05, 0.0, 3.0])
X0_CH = np.array([0.0, 0.3, 0.05, 0.0, 3.0])

results_zs = {}
for tag, dyn_t, vel_t, path_t, x0_t in [
        ("LC", dyn_lc, vel_lc, path_lc, X0_LC),
        ("CH", dyn_ch, vel_ch, path_chicane, X0_CH)]:

    # NMPC
    mpc_t = NMPCModel(dyn_model=dyn_t, vel_profile=vel_t,
                      obj_weights=dict(obj_weights),
                      L_f=L_F, t_step=T_SMPL, n_horizon=N_HORIZON)
    ev_n = EVEnergyModel(t_step=T_SMPL)
    st_n, _, _, fl_n = mpc_t.simulate(x0=x0_t.copy(), ev_energy_model=ev_n,
                                      warm_start=True, nr_steps=300,
                                      file_name=None)
    # PID-SF
    pid_t = PIDModel(dyn_model=dyn_t, vel_profile=vel_t,
                     L_f=L_F, t_step=T_SMPL,
                     Kff_kappa=1.0, Kp_lon=1.0, Ki_lon=0.02)
    ev_p = EVEnergyModel(t_step=T_SMPL)
    st_p, _, _, fl_p = pid_t.simulate(x0=x0_t.copy(), ev_energy_model=ev_p,
                                      nr_steps=300)
    # Stanley
    stan_t = StanleyModel(dyn_model=dyn_t, vel_profile=vel_t,
                          L_f=L_F, t_step=T_SMPL,
                          k_e=1.0, k_soft=1.0, K_delta=5.0,
                          Kff_kappa=1.0, Kp_lon=1.0, Ki_lon=0.02)
    ev_s = EVEnergyModel(t_step=T_SMPL)
    st_s, _, _, fl_s = stan_t.simulate(x0=x0_t.copy(), ev_energy_model=ev_s,
                                       nr_steps=300)
    # PPO - zero-shot
    st_r, ev_r, fl_r = _run_ppo_track(dyn_t, vel_t, path_t, x0_t)

    results_zs[tag] = {
        "NMPC": (float(np.sqrt(np.mean(st_n[1:, 1] ** 2))), st_n, fl_n),
        "PID":  (float(np.sqrt(np.mean(st_p[1:, 1] ** 2))), st_p, fl_p),
        "PPO":  (float(np.sqrt(np.mean(st_r[1:, 1] ** 2))) if len(st_r) > 1
                 else float("nan"), st_r, fl_r),
        "STAN": (float(np.sqrt(np.mean(st_s[1:, 1] ** 2))), st_s, fl_s),
    }

# reference S-curve numbers
r_sc = {"NMPC": float(np.sqrt(np.mean(states_nmpc[1:, 1] ** 2))),
        "PID":  float(np.sqrt(np.mean(states_pid[1:, 1] ** 2))),
        "STAN": float(np.sqrt(np.mean(states_stan[1:, 1] ** 2))),
        "PPO":  rmse_d_ppo}

print("\n" + "=" * 72)
print("MULTI-TRACK EVALUATION  (PPO: zero-shot, no retraining)")
print("=" * 72)
print(f"  {'Controller':<14} {'S-curve':>10} {'Lane-change':>13} {'Chicane':>10}")
print("  " + "-" * 52)
for ctrl in ["NMPC", "PPO", "PID", "STAN"]:
    lc = results_zs['LC'][ctrl][0]
    ch = results_zs['CH'][ctrl][0]
    fl = " (FAIL)" if (results_zs['LC'][ctrl][2] or results_zs['CH'][ctrl][2]) else ""
    print(f"  {ctrl:<14} {r_sc[ctrl]:>10.3f} {lc:>13.3f} {ch:>10.3f}{fl}")
print("=" * 72)

rmse_r_lc = results_zs["LC"]["PPO"][0]
rmse_r_ch = results_zs["CH"]["PPO"][0]
rmse_n_lc = results_zs["LC"]["NMPC"][0]
rmse_p_lc = results_zs["LC"]["PID"][0]
if rmse_r_lc < 1.0 and rmse_r_ch < 1.0:
    print("PPO generalises to ALL unseen tracks (RMSE_d < 1.0 m)")
else:
    print("PPO limited generalisation - multi-track training recommended")

In [ ]:
# Observation-noise robustness
SIGMA_LEVELS = [0.0, 0.01, 0.02, 0.05, 0.10]
noise_results = {}

print("="*55)
print("SENSOR NOISE ROBUSTNESS  (PPO eval-time only)")
print("="*55)
print(f"  {'sigma':>8} | {'RMSE_d [m]':>14} | {'E_net [kJ]':>10}")
print("  "+"-"*40)

for sigma in SIGMA_LEVELS:
    rmse_list, en_list = [], []
    for seed in range(5):
        np.random.seed(seed+100)
        _env_n = make_env(dyn_model, vel_profile, 2)
        _env_n.state = dyn_model.clip_state(X0.copy())
        _obs_n = _env_n._obs()
        _ev_n2 = EVEnergyModel(t_step=T_SMPL); _st_n2=[]
        for _ in range(500):
            _obs_in = (np.clip(_obs_n + np.random.normal(
                0, sigma, _obs_n.shape).astype(np.float32), -1., 1.)
                if sigma>0 else _obs_n)
            _act_n,_ = ppo_eval.predict(_obs_in, deterministic=True)
            _obs_n,_,_t_n,_tr_n,_ = _env_n.step(_act_n)
            _u_n = _env_n._denorm(_act_n)
            _ev_n2.step(float(_env_n.state[4]), float(_u_n[1]), 0.)
            _st_n2.append(_env_n.state.copy())
            if _t_n or _tr_n or float(_env_n.state[0])>=path.s_max-0.1: break
        if len(_st_n2)>1:
            rmse_list.append(float(np.sqrt(np.mean(np.array(_st_n2)[:,1]**2))))
            en_list.append(float(_ev_n2.e_net/1000))
    rm = float(np.mean(rmse_list)); rs = float(np.std(rmse_list))
    em = float(np.mean(en_list))
    noise_results[sigma] = {"rmse_mean":rm, "rmse_std":rs, "e_net":em}
    print(f"  {sigma:>8.3f} | {rm:>7.3f} +/- {rs:.3f} | {em:>10.2f}")
print("="*55)
# Domain-randomized noise
print("\n" + "="*55)
print("NOISE ROBUSTNESS SUMMARY")
print("="*55)
print(f"  {'sigma':>8} | {'RMSE (m)':>14}")
print("  " + "-"*28)
for sigma, res in noise_results.items():
    rm  = res['rmse_mean']
    rs  = res['rmse_std']
    flag = '' if rm < 1.0 else ' '
    print(f"  {sigma:>8.3f} | {rm:>7.3f} +/- {rs:.3f}  {flag}")
print("="*55)
print("Note: Phase-3 training now actually applies obs noise (sigma=0.02),")
print("so eval-time robustness at sigma<=0.02 is matched-in-distribution;")
print("levels above 0.05 remain out-of-distribution (see paper Sec. 5.7).")

In [ ]:
# Curriculum ablation at equal budget 
ABLATION_STEPS = 200_000 if not FAST_TEST else 8_000

ABLATIONS = {
    "A: Full (0-1-2-3-4)": {0: 50/450, 1: 120/450, 2: 80/450,
                            3: 80/450, 4: 120/450},
    "B: Phase 1 only":     {1: 1.00},
    "C: Phases 1+2":       {1: 0.60, 2: 0.40},
    "D: Phase 2 only":     {2: 1.00},
}

ablation_results = {}
print("=" * 60); print("CURRICULUM ABLATION"); print("=" * 60)

for name, phase_fracs in ABLATIONS.items():
    print(f"\nTraining: {name}")
    env_ab = Monitor(PathFollowEnv(dyn_model, vel_profile,
                                   phase=1, t_step=T_SMPL,
                                   preview=BEST_PREVIEW,
                                   energy_reward=BEST_EREWARD))
    # inject straight path
    env_ab.unwrapped._dyn_ph1  = dyn_straight
    env_ab.unwrapped._vel_ph1  = vel_straight
    env_ab.unwrapped._dyn_orig = dyn_model
    env_ab.unwrapped._vel_orig = vel_profile

    ppo_ab = PPO(env=env_ab, **PPO_KWARGS)
    active = sorted(k for k, v in phase_fracs.items() if v > 0)
    first_p = active[0]
    for phase in active:
        n = int(ABLATION_STEPS * phase_fracs[phase])
        if n <= 0:
            continue
        env_ab.unwrapped.set_phase(phase)
        ppo_ab.learn(total_timesteps=n,
                     reset_num_timesteps=(phase == first_p),
                     progress_bar=False)
    # Evaluate
    _eval = make_env(dyn_model, vel_profile, 2)
    _eval.reset(seed=0)
    _eval.state = dyn_model.clip_state(X0.copy())
    _obs_ab = _eval._obs()
    _st_ab = [_eval.state.copy()]; _ev_ab = EVEnergyModel(t_step=T_SMPL)
    for _ in range(500):
        _act_ab, _ = ppo_ab.predict(_obs_ab, deterministic=True)
        _obs_ab, _, _t_ab, _tr_ab, _ = _eval.step(_act_ab)
        _u_ab = _eval._denorm(_act_ab)
        _ev_ab.step(float(_eval.state[4]), float(_u_ab[1]), 0.)
        _st_ab.append(_eval.state.copy())
        if _t_ab or _tr_ab or float(_eval.state[0]) >= path.s_max - 0.1:
            break
    _arr_ab = np.array(_st_ab)
    rm_ab = (float(np.sqrt(np.mean(_arr_ab[1:, 1] ** 2)))
             if len(_arr_ab) > 1 else float('nan'))
    ablation_results[name] = {"rmse_d": rm_ab, "e_net": _ev_ab.e_net / 1000}
    print(f"  RMSE_d={rm_ab:.3f}m  E_net={_ev_ab.e_net/1000:.2f}kJ")

# Energy-aware ranking
best_enet = builtins.min(v["e_net"] for v in ablation_results.values())
_spread = (max(v["rmse_d"] for v in ablation_results.values())
           - min(v["rmse_d"] for v in ablation_results.values()))
print("\n" + "=" * 72); print("ABLATION SUMMARY"); print("=" * 72)
print(f"  {'Variant':<26} {'RMSE_d [m]':>11} {'E_net [kJ]':>11} {'best':>10}")
print("  " + "-" * 60)
for name, r in ablation_results.items():
    tag = "  <-- BEST" if r["e_net"] == best_enet else ""
    print(f"  {name:<26} {r['rmse_d']:>11.3f} {r['e_net']:>11.2f}{tag}")
print("=" * 72)
print(f"Reading: tracking is ~equal across variants (spread {_spread:.3f} m);")
print("the curriculum's effect is on ENERGY. The full curriculum (A) recovers")
print("the most energy (lowest E_net); no-curriculum variants track fine but")
print("waste energy: the curriculum is energy-critical, tracking-neutral.")

In [ ]:
# Dynamic single-track plant (linear tyres)
class FrenetDynamicBicycle:
    """Dynamic single-track plant, linear tires,."""

    _RHO_A, _CD_A, _AF_A = 1.225, 0.30, 2.20
    _CRR, _G = 0.01, 9.81

    def __init__(self, path, m=1500.0, Izz=3000.0, lf=2.0, lr=2.0,
                 Cf=80000.0, Cr=80000.0, t_step=0.1, v_min=1.5,
                 n_sub=10, slip_clip=0.12):
        self.path = path
        self.m, self.Izz = m, Izz
        self.lf, self.lr = lf, lr
        self.Cf, self.Cr = Cf, Cr
        self.t_step = t_step
        self.v_min = v_min
        self.n_sub = n_sub
        self.slip_clip = slip_clip
        self.wheelbase = lf + lr
        self.state_lower = np.array([0., -D_MAX, -ALPHA_MAX, -DELTA_MAX,
                                     0., -3.0, -2.0])
        self.state_upper = np.array([path.s_max, D_MAX, ALPHA_MAX, DELTA_MAX,
                                     V_MAX, 3.0, 2.0])
        self.control_lower = np.array(control_bounds[:, 0], dtype=float)
        self.control_upper = np.array(control_bounds[:, 1], dtype=float)
        self._check_stability()

    def _check_stability(self):
        """Assert RK4 stability (|lambda|*dt/n_sub <."""
        worst = 0.0
        for vx in np.linspace(self.v_min, V_MAX, 12):
            a11 = -(self.Cf + self.Cr) / (self.m * vx)
            a12 = -vx - (self.Cf * self.lf - self.Cr * self.lr) / (self.m * vx)
            a21 = -(self.Cf * self.lf - self.Cr * self.lr) / (self.Izz * vx)
            a22 = -(self.Cf * self.lf**2 + self.Cr * self.lr**2) / (self.Izz * vx)
            lam = np.max(np.abs(np.linalg.eigvals([[a11, a12], [a21, a22]])))
            worst = max(worst, lam * self.t_step / self.n_sub)
        self.rk4_margin = worst
        assert worst < 2.8, (f"RK4 UNSTABLE: lambda*dt_sub={worst:.2f} >= 2.8; "
                             f"increase n_sub (current {self.n_sub}).")

    def _f(self, x, u):
        s, d, alpha, delta, vx, vy, r = x
        u1, u2 = u
        kappa = float(self.path.curvature(float(np.clip(s, 0., self.path.s_max))))
        vx_s = max(vx, self.v_min)

        af = float(np.clip(delta - (vy + self.lf * r) / vx_s,
                           -self.slip_clip, self.slip_clip))
        ar = float(np.clip(-(vy - self.lr * r) / vx_s,
                           -self.slip_clip, self.slip_clip))
        Fyf, Fyr = self.Cf * af, self.Cr * ar

        f_res = (0.5 * self._RHO_A * self._CD_A * self._AF_A * vx * vx
                 + self.m * self._G * self._CRR) if vx > 0.05 else 0.0

        denom = 1.0 - d * kappa
        if abs(denom) < 0.3:
            denom = 0.3 * (np.sign(denom) if denom != 0 else 1.0)
        s_dot = (vx * np.cos(alpha) - vy * np.sin(alpha)) / denom

        return np.array([
            s_dot,
            vx * np.sin(alpha) + vy * np.cos(alpha),
            r - kappa * s_dot,
            u1,
            u2 - f_res / self.m + r * vy,
            (Fyf * np.cos(delta) + Fyr) / self.m - r * vx,
            (self.lf * Fyf * np.cos(delta) - self.lr * Fyr) / self.Izz,
        ])

    def clip_state(self, x):
        return np.clip(np.asarray(x, float), self.state_lower, self.state_upper)

    def clip_control(self, u):
        return np.clip(np.asarray(u, float), self.control_lower, self.control_upper)

    def rk4_step(self, x, u, clipped=True):
        x = np.asarray(x, float)
        u = self.clip_control(u)
        h = self.t_step / self.n_sub
        for _ in range(self.n_sub):
            k1 = self._f(x, u)
            k2 = self._f(x + 0.5 * h * k1, u)
            k3 = self._f(x + 0.5 * h * k2, u)
            k4 = self._f(x + h * k3, u)
            x = x + h * (k1 + 2 * k2 + 2 * k3 + k4) / 6.0
            # per-substep clamp
            x[5] = np.clip(x[5], self.state_lower[5], self.state_upper[5])
            x[6] = np.clip(x[6], self.state_lower[6], self.state_upper[6])
        return self.clip_state(x) if clipped else x

V_MAX_DYN = 8.0
vel6 = ReferenceVelocityGenerator(
    path, max_speed=V_MAX_DYN,
    max_lateral_acceleration=A_LAT_MAX,
    max_deceleration=A_LNG_MAX,
    nr_path_samples=100, filter_window=3)

plant_dyn = FrenetDynamicBicycle(path, t_step=T_SMPL)
X0_DYN7 = np.array([0.0, 0.5, 0.05, 0.0, 3.0, 0.0, 0.0])
print(f"Dynamic bicycle plant ready | wheelbase={plant_dyn.wheelbase} m | "
      f"Izz={plant_dyn.Izz} | n_sub={plant_dyn.n_sub} | "
      f"RK4 margin={plant_dyn.rk4_margin:.2f} (<2.8 OK) | v<={V_MAX_DYN} m/s")

class DynPathFollowEnv(PathFollowEnv):
    """PathFollowEnv whose plant is the."""

    def __init__(self, kin_model, vel_p, plant, phase=2, t_step=0.1,
                 preview=False, energy_reward=False):
        super().__init__(kin_model, vel_p, phase=phase, t_step=t_step,
                         preview=preview, energy_reward=energy_reward)
        self.plant = plant
        self._x7 = None

    def reset(self, seed=None, options=None):
        obs, info = super().reset(seed=seed)
        self._x7 = np.concatenate([self.state, [0.0, 0.0]])
        return obs, info

    def set_state7(self, x7):
        self._x7 = self.plant.clip_state(np.asarray(x7, float))
        self.state = self._x7[:5].copy()
        return self._obs()

    def step(self, action):
        u = self._denorm(action)
        s_prev = float(self.state[0])
        self._x7 = self.plant.rk4_step(self._x7, u)
        self.state = self._x7[:5].copy()
        self._steps += 1
        r = self._reward(s_prev, u)
        s, d = float(self.state[0]), float(self.state[1])
        oob = abs(d) >= self.d_max - 0.1
        terminated = bool(s >= self.s_max - 0.1 or oob)
        truncated = bool(self._steps >= self._MAX)
        if oob:
            r -= 5.0
        return self._obs(), r, terminated, truncated, {"failed": oob}

print("DynPathFollowEnv ready (7D plant, 5D controller view)")

In [ ]:
# Dynamic-model LQR baseline 
def _rmse_d(states):
    a = np.asarray(states, dtype=float)  # force float: kills
    return float(np.sqrt(np.mean(a[1:, 1] ** 2))) if len(a) > 1 else float("nan")

def _regen_frac(ev):
    return 100.0 * ev.e_recovered / max(ev.e_propulsion, 1e-9)

def _status(states_hist, ev, reached_end, went_oob):
    if went_oob:
        return "off-road"
    a = np.asarray(states_hist)
    if len(a) >= 20:
        ub = plant_dyn.state_upper
        tv, tr = a[-20:, 5], a[-20:, 6]
        if np.all(np.abs(np.abs(tv) - ub[5]) < 0.02) or \
           np.all(np.abs(np.abs(tr) - ub[6]) < 0.02):
            return "tire-sat"
    if _regen_frac(ev) > 100.0 or ev.e_propulsion < 1e-6:
        return "num-invalid"
    return "ok" if reached_end else "incomplete"

from scipy.linalg import solve_continuous_are

class DynamicLQRController:
    """Speed-scheduled lateral LQR on the."""

    def __init__(self, dyn_plant, vel_profile, t_step=0.1,
                 Q=(1.0, 0.0, 1.0, 0.0), R=8.0, Ki=0.4,
                 Kp_lon=1.0, Ki_lon=0.02):
        self.plant = dyn_plant
        self.vel_profile = vel_profile
        self.t_step = t_step
        self.Q = np.diag(Q)
        self.R = np.array([[float(R)]])
        self.Ki = Ki
        self.Kp_lon, self.Ki_lon = Kp_lon, Ki_lon
        self.m, self.Iz = dyn_plant.m, dyn_plant.Izz
        self.lf, self.lr = dyn_plant.lf, dyn_plant.lr
        self.Cf, self.Cr = dyn_plant.Cf, dyn_plant.Cr
        self.L = self.lf + self.lr
        self.reset()

    def reset(self):
        self._e1_prev = 0.0
        self._e2_prev = 0.0
        self._ei = 0.0
        self._lon_int = 0.0

    def _gain(self, vx):
        vx = max(vx, 1.5)
        m, Iz, lf, lr, Cf, Cr = self.m, self.Iz, self.lf, self.lr, self.Cf, self.Cr
        A = np.array([
            [0, 1, 0, 0],
            [0, -(2*Cf+2*Cr)/(m*vx), (2*Cf+2*Cr)/m, (-2*Cf*lf+2*Cr*lr)/(m*vx)],
            [0, 0, 0, 1],
            [0, -(2*Cf*lf-2*Cr*lr)/(Iz*vx), (2*Cf*lf-2*Cr*lr)/Iz,
             -(2*Cf*lf**2+2*Cr*lr**2)/(Iz*vx)],
        ])
        B = np.array([[0], [2*Cf/m], [0], [2*Cf*lf/Iz]])
        P = solve_continuous_are(A, B, self.Q, self.R)
        return (np.linalg.inv(self.R) @ B.T @ P).flatten()

    def steering_angle(self, state5):
        """Desired front steering angle from."""
        s, d, alpha, delta, vx = [float(v) for v in state5]
        vx = max(vx, 0.1)
        s_c = float(self.plant.path.clip(s))
        kappa = float(self.plant.path.curvature(s_c))
        K = self._gain(vx)
        # steady-state heading
        e2ss = -self.lr*kappa + self.lf*self.m*vx*vx*kappa/(2*self.Cr*self.L)
        e1d = (d - self._e1_prev) / self.t_step
        e2d = (alpha - self._e2_prev) / self.t_step
        self._e1_prev, self._e2_prev = d, alpha
        # conditional integral
        if abs(d) < 0.3 and abs(e1d) < 0.3:
            self._ei = float(np.clip(self._ei + d*self.t_step, -1.0, 1.0))
        else:
            self._ei *= 0.9
        xe = np.array([d, e1d, alpha - e2ss, e2d])
        delta_cmd = -K @ xe + self.L*kappa - self.Ki*self._ei
        return float(np.clip(delta_cmd, -DELTA_MAX, DELTA_MAX)), s_c

    def longitudinal(self, state5, s_c):
        vx = max(float(state5[4]), 0.1)
        v_ref = float(self.vel_profile.get_maximum_speed(s_c))
        e_v = v_ref - vx
        self._lon_int = float(np.clip(self._lon_int + e_v*self.t_step, -5, 5))
        return self.Kp_lon*e_v + self.Ki_lon*self._lon_int

def run_lqr_on_dynamic(ctrl, x0_7, nr_steps=400, k_delta=8.0):
    """Closed loop: LQR outputs a."""
    ctrl.reset()
    x7 = plant_dyn.clip_state(np.asarray(x0_7, float))
    st, ev, status = [x7.copy()], EVEnergyModel(t_step=T_SMPL), "ok"
    reached, oob = False, False
    for _ in range(nr_steps):
        delta_cmd, s_c = ctrl.steering_angle(x7[:5])
        u2 = ctrl.longitudinal(x7[:5], s_c)
        u1 = k_delta * (delta_cmd - float(x7[3]))  # inner steering loop
        u = plant_dyn.clip_control(np.array([u1, u2]))
        x7 = plant_dyn.rk4_step(x7, u)
        st.append(x7.copy()); ev.step(float(x7[4]), float(u[1]), 0.)
        if float(x7[0]) >= path.s_max - 0.1:
            reached = True; break
        if abs(float(x7[1])) >= D_MAX - 0.1:
            oob = True; break
    status = _status(st, ev, reached, oob)
    return np.array(st), ev, status

# Step-response validation
_lqr = DynamicLQRController(plant_dyn, vel6, t_step=T_SMPL)
# straight-path settling
_x = np.array([0.0, 1.0, 0.0, 0.0, 5.0, 0.0, 0.0])
_lqr.reset(); _hist = [_x.copy()]
for _ in range(150):
    _dc, _sc = _lqr.steering_angle(_x[:5])
    # force kappa=0
    _u1 = 8.0*(_dc - float(_x[3]))
    _x = plant_dyn.rk4_step(_x, plant_dyn.clip_control(np.array([_u1, 0.0])))
    _hist.append(_x.copy())
_hist = np.array(_hist)
_settle = next((k*T_SMPL for k in range(len(_hist))
                if np.all(np.abs(_hist[k:, 1]) < 0.05)), float("nan"))
print(f"Dynamic LQR step response (d0=1m, v=5m/s): "
      f"settling(|d|<0.05m) = {_settle:.2f} s, "
      f"overshoot = {max(0, np.max(-_hist[:,1])):.3f} m")
print("Dynamic LQR baseline ready (error-state, curvature FF, "
      "conditional integral)")

In [ ]:
# Dynamic PID-SF (re-tuned poles) 
class PIDModelDyn(PIDModel):
    """Slower pole set matched to."""
    DESIRED_POLES = np.array([-0.8, -1.4, -2.5])

    def _lateral_gains(self, v):
        v = max(v, 1.5)
        ratio = self.V_NOM / v
        return self._Kd_nom * ratio * ratio, self._Kalpha_nom * ratio, self._Kdelta

def _rmse_d(states):
    a = np.asarray(states, dtype=float)  # force float: kills
    return float(np.sqrt(np.mean(a[1:, 1] ** 2))) if len(a) > 1 else float("nan")

def _regen_frac(ev):
    return 100.0 * ev.e_recovered / max(ev.e_propulsion, 1e-9)

def _status(states_hist, ev, reached_end, went_oob):
    """Legible status for the dynamic."""
    if went_oob:
        return "off-road"
    a = np.asarray(states_hist)
    if len(a) >= 20:
        ub = plant_dyn.state_upper
        tv, tr = a[-20:, 5], a[-20:, 6]
        if np.all(np.abs(np.abs(tv) - ub[5]) < 0.02) or \
           np.all(np.abs(np.abs(tr) - ub[6]) < 0.02):
            return "tire-sat"
    if _regen_frac(ev) > 100.0 or ev.e_propulsion < 1e-6:
        return "num-invalid"  # impossible energy ->
    return "ok" if reached_end else "incomplete"

def run_ctrl_on_dynamic(ctrl, x0_7, nr_steps=400):
    if hasattr(ctrl, "reset"):
        ctrl.reset()
    x7 = plant_dyn.clip_state(np.asarray(x0_7, float))
    st, ev = [x7.copy()], EVEnergyModel(t_step=T_SMPL)
    reached, oob = False, False
    for _ in range(nr_steps):
        u = ctrl.compute_control(x7[:5])
        x7 = plant_dyn.rk4_step(x7, u)
        st.append(x7.copy()); ev.step(float(x7[4]), float(u[1]), 0.)
        if float(x7[0]) >= path.s_max - 0.1:
            reached = True; break
        if abs(float(x7[1])) >= D_MAX - 0.1:
            oob = True; break
    return np.array(st), ev, _status(st, ev, reached, oob)

def run_nmpc_on_dynamic(mpc, x0_7, nr_steps=400):
    x7 = plant_dyn.clip_state(np.asarray(x0_7, float))
    x5 = x7[:5].copy()
    mpc._optimizer.x0 = x5; mpc._optimizer.set_initial_guess()
    st, ev, tms = [x7.copy()], EVEnergyModel(t_step=T_SMPL), []
    reached, oob = False, False
    for _ in range(nr_steps):
        t0 = time.perf_counter()
        u0 = mpc._optimizer.make_step(x5.reshape(-1, 1))
        tms.append(time.perf_counter() - t0)
        u = np.asarray(u0, float).reshape(-1)
        x7 = plant_dyn.rk4_step(x7, u); x5 = x7[:5].copy()
        st.append(x7.copy()); ev.step(float(x7[4]), float(u[1]), tms[-1])
        if float(x7[0]) >= path.s_max - 0.1:
            reached = True; break
        if abs(float(x7[1])) >= D_MAX - 0.1:
            oob = True; break
    return np.array(st), ev, np.array(tms), _status(st, ev, reached, oob)

def run_ppo_on_dynamic(model, x0_7, nr_steps=500):
    env_d = DynPathFollowEnv(dyn_model, vel6, plant_dyn, phase=2,
                             t_step=T_SMPL, preview=BEST_PREVIEW,
                             energy_reward=BEST_EREWARD)
    env_d.reset(seed=0)
    obs = env_d.set_state7(np.asarray(x0_7, float))
    st, ev = [env_d._x7.copy()], EVEnergyModel(t_step=T_SMPL)
    reached, oob = False, False
    for _ in range(nr_steps):
        act, _ = model.predict(obs, deterministic=True)
        obs, _, term, trunc, info = env_d.step(act)
        u = env_d._denorm(act)
        st.append(env_d._x7.copy())
        ev.step(float(env_d._x7[4]), float(u[1]), 0.)
        if info.get("failed", False):
            oob = True
        if float(env_d._x7[0]) >= path.s_max - 0.1:
            reached = True
        if term or trunc:
            break
    return np.array(st), ev, _status(st, ev, reached, oob)

# PID step-response
pid_dyn = PIDModelDyn(dyn_model=dyn_model, vel_profile=vel6, L_f=L_F,
                      t_step=T_SMPL, Kff_kappa=1.0, Kp_lon=1.0, Ki_lon=0.02)
print(f"PID-Dyn gains: Kd={pid_dyn._Kd_nom:.2f}  Ka={pid_dyn._Kalpha_nom:.2f} "
      f" Kdelta={pid_dyn._Kdelta:.2f}  (poles {list(PIDModelDyn.DESIRED_POLES)})")
_x7s = np.array([0., 0.5, 0., 0., 4.0, 0., 0.])
_sts, _, _ = run_ctrl_on_dynamic(pid_dyn, _x7s, nr_steps=150)
_settle = next((k * T_SMPL for k in range(len(_sts))
                if np.all(np.abs(_sts[k:, 1]) < 0.05)), float("nan"))
print(f"Step response (d0=0.5m, v=4m/s) on DYNAMIC plant: "
      f"settling(|d|<0.05m) = {_settle:.2f} s  "
      f"(peak |vy|={np.max(np.abs(_sts[:,5])):.2f}, |r|={np.max(np.abs(_sts[:,6])):.2f})")

# Benchmark runs
print("\nRunning dynamic benchmark...")
mpc6 = NMPCModel(dyn_model=dyn_model, vel_profile=vel6,
                 obj_weights=dict(obj_weights), L_f=L_F,
                 t_step=T_SMPL, n_horizon=N_HORIZON)
st_n7, ev_n7, tm_n7, stt_n7 = run_nmpc_on_dynamic(mpc6, X0_DYN7)
st_p7, ev_p7, stt_p7 = run_ctrl_on_dynamic(pid_dyn, X0_DYN7)

# Dynamic-model LQR baseline
lqr_dyn = DynamicLQRController(plant_dyn, vel6, t_step=T_SMPL)
st_l7, ev_l7, stt_l7 = run_lqr_on_dynamic(lqr_dyn, X0_DYN7)

stan_dyn = StanleyModel(dyn_model=dyn_model, vel_profile=vel6, L_f=L_F,
                        t_step=T_SMPL, k_e=0.5, k_soft=1.5, K_delta=3.0,
                        Kff_kappa=1.0, Kp_lon=1.0, Ki_lon=0.02)
st_s7, ev_s7, stt_s7 = run_ctrl_on_dynamic(stan_dyn, X0_DYN7)
st_z7, ev_z7, stt_z7 = run_ppo_on_dynamic(ppo_eval, X0_DYN7)

print("Fine-tuning PPO on the dynamic plant (100k steps)...")
env_ft = Monitor(DynPathFollowEnv(dyn_model, vel6, plant_dyn, phase=2,
                                  t_step=T_SMPL, preview=BEST_PREVIEW,
                                  energy_reward=BEST_EREWARD))
ppo_ft = PPO.load("ppo_pathfollow", env=env_ft, device="cpu")
env_ft.unwrapped.set_phase(2)
ppo_ft.learn(total_timesteps=60_000, reset_num_timesteps=False, progress_bar=False)
env_ft.unwrapped.set_phase(4)
ppo_ft.learn(total_timesteps=40_000, reset_num_timesteps=False, progress_bar=False)
ppo_ft.save("ppo_pathfollow_dyn")
st_f7, ev_f7, stt_f7 = run_ppo_on_dynamic(ppo_ft, X0_DYN7)

# Table
print("\n" + "=" * 80)
print(f"DYNAMIC APPENDIX  (linear tyres Cf=Cr=80 kN/rad, "
      f"{plant_dyn.wheelbase:.0f} m wheelbase, v <= {V_MAX_DYN:.0f} m/s)")
print("=" * 80)
print(f"  {'Controller':<24} {'RMSE_d [m]':>11} {'E_net [kJ]':>11} "
      f"{'Regen %':>9} {'Status':>12}")
print("  " + "-" * 72)
for nm, stx, evx, stt in [
        ("NMPC (kin. model)", st_n7, ev_n7, stt_n7),
        ("LQR (dyn. model)", st_l7, ev_l7, stt_l7),
        ("PID-SF (kin.-designed)", st_p7, ev_p7, stt_p7),
        ("Stanley (re-tuned)", st_s7, ev_s7, stt_s7),
        ("PPO zero-shot", st_z7, ev_z7, stt_z7),
        ("PPO fine-tuned 100k", st_f7, ev_f7, stt_f7)]:
    rf = _regen_frac(evx)
    rf_str = f"{rf:>7.1f}%" if rf <= 100.0 else "  n/a  "
    en_str = f"{evx.e_net/1000:>11.2f}" if stt != "num-invalid" else f"{'invalid':>11}"
    print(f"  {nm:<24} {_rmse_d(stx):>11.3f} {en_str} {rf_str} {stt:>12}")
print("=" * 80)
print("Status: ok=reached end | off-road=|d|>=d_max-0.1 | tire-sat=vy/r pinned")
print("        (exited linear-tire regime) | num-invalid=impossible energy (diverged)")
try:
    print(f"PPO fine-tuned peaks: |vy|={np.max(np.abs(st_f7[:,5])):.2f} m/s, "
          f"|r|={np.max(np.abs(st_f7[:,6])):.2f} rad/s, "
          f"|a_lat|~{np.max(np.abs(st_f7[:,4]*st_f7[:,6])):.2f} m/s^2")
except Exception:
    pass

In [ ]:
# Compiled NMPC timing (pure CasADi + JIT) 
import casadi as ca

N_H, DT = N_HORIZON, T_SMPL
_w = mpc._w  # already-normalized weights

x_sym = ca.SX.sym("x", 5); u_sym = ca.SX.sym("u", 2); k_sym = ca.SX.sym("k")

def _fkin(x, u, k):
    s, d, al, de, v = ca.vertsplit(x)
    u1, u2 = ca.vertsplit(u)
    sdot = v * ca.cos(al) / (1 - d * k)
    return ca.vertcat(sdot, v * ca.sin(al),
                      v * ca.tan(de) / L_F - k * sdot, u1, u2)

def _rk4(x, u, k):
    k1 = _fkin(x, u, k); k2 = _fkin(x + DT/2*k1, u, k)
    k3 = _fkin(x + DT/2*k2, u, k); k4 = _fkin(x + DT*k3, u, k)
    return x + DT/6*(k1 + 2*k2 + 2*k3 + k4)

X = ca.SX.sym("X", 5, N_H + 1); U = ca.SX.sym("U", 2, N_H)
P = ca.SX.sym("P", 5 + 2 * (N_H + 1))  # x0, kappa_pred, vref_pred
kap = P[5:5+N_H+1]; vrf = P[5+N_H+1:]

g_list = [X[:, 0] - P[:5]]; J = 0
for j in range(N_H):
    J += (_w["d"] * X[1, j]**2 + _w["alpha"] * X[2, j]**2
          + _w["v"] * (X[4, j] - vrf[j])**2
          + _w["u1"] * U[0, j]**2 + _w["u2"] * U[1, j]**2)
    g_list.append(X[:, j+1] - _rk4(X[:, j], U[:, j], kap[j]))
g = ca.vertcat(*g_list)
z = ca.vertcat(ca.reshape(X, -1, 1), ca.reshape(U, -1, 1))

opts = {"ipopt.max_iter": 20, "ipopt.print_level": 0, "print_time": 0,
        "ipopt.tol": 1e-6,
        "jit": True, "compiler": "shell",
        "jit_options": {"flags": ["-O2"], "verbose": False}}
print("Building + JIT-compiling NLP (one-time, ~30-90 s)...")
t0 = time.perf_counter()
solver = ca.nlpsol("solver", "ipopt", {"x": z, "f": J, "g": g, "p": P}, opts)
print(f"  compiled in {time.perf_counter()-t0:.1f} s")

# bounds
lbx = np.concatenate([np.tile(dyn_model.state_lower, N_H+1),
                      np.tile(dyn_model.control_lower, N_H)])
ubx = np.concatenate([np.tile(dyn_model.state_upper, N_H+1),
                      np.tile(dyn_model.control_upper, N_H)])
lbg = ubg = np.zeros(5 * (N_H + 1))

def _params(x5):
    s_pred = np.clip(np.linspace(x5[0], x5[0] + V_MAX*N_H*DT, N_H+1),
                     0., path.s_max)
    return np.concatenate([x5, path.curvature(s_pred),
                           vel_profile.get_maximum_speed(s_pred)])

# closed loop
x5 = np.array(X0, float); z0 = np.concatenate([np.tile(x5, N_H+1),
                                               np.zeros(2*N_H)])
lam = None; tms_c = []
ev_c = EVEnergyModel(t_step=T_SMPL); st_c = [x5.copy()]
for i in range(200):
    t0 = time.perf_counter()
    kw = dict(x0=z0, p=_params(x5), lbx=lbx, ubx=ubx, lbg=lbg, ubg=ubg)
    if lam is not None:
        kw["lam_x0"], kw["lam_g0"] = lam
    sol = solver(**kw)
    tms_c.append(time.perf_counter() - t0)
    zopt = np.array(sol["x"]).reshape(-1)
    u0 = zopt[5*(N_H+1):5*(N_H+1)+2]
    x5 = dyn_model.rk4_step(x5, u0, clipped=True, nonlinear=True)
    st_c.append(x5.copy()); ev_c.step(float(x5[4]), float(u0[1]), tms_c[-1])
    # warm start
    Xo = zopt[:5*(N_H+1)].reshape(N_H+1, 5); Uo = zopt[5*(N_H+1):].reshape(N_H, 2)
    z0 = np.concatenate([np.vstack([Xo[1:], Xo[-1:]]).reshape(-1),
                         np.vstack([Uo[1:], Uo[-1:]]).reshape(-1)])
    lam = (sol["lam_x"], sol["lam_g"])
    if float(x5[0]) >= path.s_max - 0.1:
        break

tms_c = np.array(tms_c) * 1000
rmse_c = float(np.sqrt(np.mean(np.array(st_c)[1:, 1]**2)))
print("\n" + "=" * 66)
print("COMPILED NMPC (pure CasADi + JIT, warm start)")
print("=" * 66)
print(f"  RMSE_d = {rmse_c:.3f} m   (do-mpc baseline: 0.474 m)")
print(f"  Solve: avg {tms_c.mean():.1f} ms | p95 {np.percentile(tms_c,95):.1f} ms"
      f" | max {tms_c.max():.1f} ms   (baseline avg 114.2 / p95 145.5)")
print(f"  Real-time (p95 < {T_SMPL*1000:.0f} ms): "
      + ("YES" if np.percentile(tms_c,95) < T_SMPL*1000 else "NO"))
_t_ppo_ms = float(np.mean(tm_ppo) * 1000)  # measured PPO inference
print(f"  Measured PPO solve time: {_t_ppo_ms:.3f} ms")
print(f"  PPO is {tms_c.mean()/_t_ppo_ms:.0f}x faster than the compiled NMPC")
print("=" * 66)

In [ ]:
# Predictive safety filter
D_SAFE = 4.5  # [m] enforce |d|
N_LOOK = 4  # rollout horizon [steps]
N_GRID = 9

def safety_filter(state5, u_prop):
    """Returns (u_safe, intervened)."""
    u_prop = np.asarray(u_prop, float)
    cands = np.linspace(-dyn_model.control_upper[0],
                        dyn_model.control_upper[0], N_GRID)
    cands = cands[np.argsort(np.abs(cands - u_prop[0]))]
    for c in cands:
        x = np.asarray(state5, float).copy()
        ok = True
        for _ in range(N_LOOK):
            x = dyn_model.rk4_step(x, np.array([c, u_prop[1]]),
                                   clipped=True, nonlinear=True)
            if abs(float(x[1])) >= D_SAFE:
                ok = False; break
        if ok:
            return np.array([c, u_prop[1]]), bool(abs(c - u_prop[0]) > 1e-9)
    # no admissible candidate
    d_now = float(state5[1])
    return np.array([-np.sign(d_now) * dyn_model.control_upper[0],
                     -dyn_model.control_upper[1]]), True

def run_ppo_filtered(x0_5, use_filter=True, max_steps=500):
    env_f = make_env(dyn_model, vel_profile, 2)
    env_f.reset(seed=0)
    env_f.state = dyn_model.clip_state(np.asarray(x0_5, float))
    obs = env_f._obs()
    n_int, viol, st = 0, False, [env_f.state.copy()]
    for _ in range(max_steps):
        act, _ = ppo_eval.predict(obs, deterministic=True)
        u = env_f._denorm(act)
        if use_filter:
            u_s, inter = safety_filter(env_f.state, u)
            n_int += int(inter)
            act = np.array([u_s[0] / env_f.u1_max, u_s[1] / env_f.u2_max],
                           dtype=np.float32)
        obs, _, term, trunc, info = env_f.step(act)
        st.append(env_f.state.copy())
        viol = viol or abs(float(env_f.state[1])) >= 4.9 - 1e-6
        if term or trunc or float(env_f.state[0]) >= path.s_max - 0.1:
            break
    arr = np.array(st)
    return float(np.sqrt(np.mean(arr[1:, 1]**2))), n_int, viol, len(arr) - 1

print("(a) Nominal N=30 LHS ICs, filter ON:")
_rm_f, _int_f = [], 0
for x0_ic in ICS_30:
    r, ni, vi, ns = run_ppo_filtered(x0_ic, use_filter=True)
    _rm_f.append(r); _int_f += ni
print(f"  RMSE_d = {np.nanmean(_rm_f):.3f} +/- {np.nanstd(_rm_f):.3f} m  "
      f"(unfiltered N=30: {np.nanmean(res30['PPO']['rmse_d']):.3f} +/- "
      f"{np.nanstd(res30['PPO']['rmse_d']):.3f})")
print(f"  Total interventions across 30 runs: {_int_f}")

print("\n(b) Extreme ICs (|d0| up to 4.2 m):")
EXTREME_ICS = [np.array([0., d0, a0, 0., v0])
               for d0 in (-4.2, -3.5, -2.5, 2.5, 3.5, 4.2)
               for a0, v0 in ((0.15, 4.0), (-0.15, 6.0))]
print(f"  {'':<10} {'violations':>11} {'mean interv.':>13} {'mean RMSE':>10}")
for tag, uf in (("filter ON", True), ("filter OFF", False)):
    _v, _i, _r = 0, [], []
    for x0e in EXTREME_ICS:
        r, ni, vi, ns = run_ppo_filtered(x0e, use_filter=uf)
        _v += int(vi); _i.append(ni); _r.append(r)
    print(f"  {tag:<10} {_v:>8}/{len(EXTREME_ICS)} {np.mean(_i):>13.1f} "
          f"{np.nanmean(_r):>10.3f}")
print("\nPaper: filter is inactive in nominal operation (results unchanged)")
print("Reading: the filter is effectively inactive in nominal operation,")
print("so all reported results are unchanged by its presence. Under")
print("extreme initialisation it intervenes frequently but does NOT")
print("remove every corridor violation; it is a projection heuristic,")
print("not a formal safety certificate.")

In [ ]:
# Random-track generalisation
def _f(x):
    try:
        import casadi as _ca
        if isinstance(x, _ca.DM):
            return float(x.full().ravel()[0]) if x.numel() > 0 else float("nan")
    except Exception:
        pass
    try:
        arr = np.asarray(x, dtype=float).ravel()
        return float(arr[0]) if arr.size == 1 else float(np.nanmean(arr))
    except Exception:
        return float(x)

# self-contained guard
if "_rmse_d" not in dir():
    def _rmse_d(states):
        a = np.asarray(states)
        return float(np.sqrt(np.mean(a[1:, 1] ** 2))) if len(a) > 1 else float("nan")
def make_random_track(seed, length=90.0, n_wp=8, y_amp=5.0):
    rng = np.random.default_rng(seed)
    xs = np.linspace(0., length, n_wp)
    for _ in range(20):
        y = np.cumsum(rng.uniform(-1., 1., n_wp))
        y = (y - y.mean())
        y = y / max(np.max(np.abs(y)), 1e-9) * y_amp
        wp = np.column_stack([xs, y])
        p = ArclengthSpline2D(wp, filter_pts=False, nr_resamples=10)
        smax_chk = np.linspace(0., p.s_max, 300)
        kmax = float(np.max(np.abs(np.asarray(p.curvature(smax_chk), dtype=float))))
        if kmax <= 0.95 * KAPPA_SCALE:
            return p, kmax
        y_amp *= 0.8
    return p, kmax

def _mk_models(p):
    sb = np.array([[0, p.s_max], [-D_MAX, D_MAX], [-ALPHA_MAX, ALPHA_MAX],
                   [-DELTA_MAX, DELTA_MAX], [0, V_MAX]])
    dm = FrenetDynamicModel(path=p, state_bounds=sb,
                            control_bounds=control_bounds,
                            L_f=L_F, t_step=T_SMPL)
    vp = ReferenceVelocityGenerator(p, max_speed=V_MAX,
                                    max_lateral_acceleration=A_LAT_MAX,
                                    max_deceleration=A_LNG_MAX,
                                    nr_path_samples=100, filter_window=3)
    return dm, vp

N_TRACKS = 30
X0_RT = np.array([0.0, 0.3, 0.05, 0.0, 3.0])
rt = {c: {"rmse": [], "fail": []} for c in ["PPO", "PID", "STAN", "NMPC"]}
kmax_list = []

for ti in range(N_TRACKS):
    p_t, km = make_random_track(seed=1000 + ti)
    kmax_list.append(_f(km))
    dm_t, vp_t = _mk_models(p_t)

    # PPO zero-shot
    env_t = make_env(dm_t, vp_t, 2); env_t.reset(seed=0)
    env_t.state = dm_t.clip_state(X0_RT.copy()); obs_t = env_t._obs()
    st_t, fl_t = [env_t.state.copy()], False
    for _ in range(700):
        a_t, _ = ppo_eval.predict(obs_t, deterministic=True)
        obs_t, _, tm_t, tr_t, inf_t = env_t.step(a_t)
        st_t.append(env_t.state.copy())
        fl_t = fl_t or bool(inf_t.get("failed", False))
        if tm_t or tr_t or float(env_t.state[0]) >= p_t.s_max - 0.1:
            break
    rt["PPO"]["rmse"].append(_f(_rmse_d(st_t)))
    rt["PPO"]["fail"].append(bool(fl_t))

    # PID / Stanley
    for nm, cls, kw in (("PID", PIDModel, {}),
                        ("STAN", StanleyModel,
                         dict(k_e=1.0, k_soft=1.0, K_delta=5.0))):
        c_t = cls(dyn_model=dm_t, vel_profile=vp_t, L_f=L_F, t_step=T_SMPL,
                  Kff_kappa=1.0, Kp_lon=1.0, Ki_lon=0.02, **kw)
        s_t, _, _, f_t = c_t.simulate(x0=X0_RT.copy(),
                                      ev_energy_model=EVEnergyModel(T_SMPL),
                                      nr_steps=400)
        rt[nm]["rmse"].append(_f(_rmse_d(s_t)))
        rt[nm]["fail"].append(bool(f_t))

    # NMPC on ALL
    if True:
        m_t = NMPCModel(dyn_model=dm_t, vel_profile=vp_t,
                        obj_weights=dict(obj_weights), L_f=L_F,
                        t_step=T_SMPL, n_horizon=N_HORIZON)
        s_t, _, _, f_t = m_t.simulate(x0=X0_RT.copy(),
                                      ev_energy_model=EVEnergyModel(T_SMPL),
                                      warm_start=True, nr_steps=400,
                                      file_name=None)
        rt["NMPC"]["rmse"].append(_f(_rmse_d(s_t)))
        rt["NMPC"]["fail"].append(bool(f_t))
    print(f"  track {ti+1:2d}/{N_TRACKS} (kmax={km:.3f}): done")

print("\n" + "=" * 70)
print(f"RANDOM-TRACK GENERALIZATION  ({N_TRACKS} unseen splines, "
      f"kmax {_f(np.min(kmax_list)):.3f}-{_f(np.max(kmax_list)):.3f} 1/m)")
print("=" * 70)
print(f"  {'Controller':<10} {'tracks':>7} {'RMSE_d [m]':^22} {'Fail':>7}")
print("  " + "-" * 54)
for c in ["PPO", "PID", "STAN", "NMPC"]:
    r = np.array([float(x) for x in rt[c]["rmse"]], dtype=float)
    f = [bool(x) for x in rt[c]["fail"]]
    if len(r) == 0:
        continue
    mean_r = _f(np.nanmean(np.array([_f(x) for x in rt[c]["rmse"]])))
    std_r  = _f(np.nanstd(np.array([_f(x) for x in rt[c]["rmse"]])))
    n_fail = int(sum(1 for x in f if bool(x)))
    print(f"  {c:<10} {len(r):>7} {mean_r:>10.3f} +/- {std_r:>6.3f}      {n_fail:>3}/{len(f)}")
print("=" * 70)
print("Paper: replaces the 3-track evaluation with a track DISTRIBUTION.")

In [ ]:
# Road-grade robustness
def _f(x):
    try:
        import casadi as _ca
        if isinstance(x, _ca.DM):
            return float(x.full().ravel()[0]) if x.numel() > 0 else float("nan")
    except Exception:
        pass
    try:
        arr = np.asarray(x, dtype=float).ravel()
        return float(arr[0]) if arr.size == 1 else float(np.nanmean(arr))
    except Exception:
        return float(x)

# self-contained guard
if "_rmse_d" not in dir():
    def _rmse_d(states):
        a = np.asarray(states)
        return float(np.sqrt(np.mean(a[1:, 1] ** 2))) if len(a) > 1 else float("nan")
THETA_AMP = np.deg2rad(3.0)  # ~5.2% peak grade
_G = 9.81

def theta_of_s(s):
    return THETA_AMP * np.sin(2. * np.pi * float(s) / path.s_max)

def _grade_step(x5, u):
    """Kinematic plant + grade: u2_eff."""
    u = dyn_model.clip_control(np.asarray(u, float))
    u_eff = np.array([u[0], u[1] - _G * np.sin(theta_of_s(x5[0]))])
    xn = dyn_model.rk4_step(x5, u_eff, clipped=False, nonlinear=True)
    return dyn_model.clip_state(xn), u

def _ev_grade_step(ev, x5, u, t_solve):
    ev._EV_THETA = theta_of_s(x5[0])
    ev.step(float(x5[4]), float(u[1]), t_solve)

res_grade = {}

# PPO
env_g = make_env(dyn_model, vel_profile, 2); env_g.reset(seed=0)
env_g.state = dyn_model.clip_state(X0.copy()); obs_g = env_g._obs()
x5 = env_g.state.copy(); st_g = [x5.copy()]; ev_g = EVEnergyModel(T_SMPL)
for _ in range(500):
    a_g, _ = ppo_eval.predict(obs_g, deterministic=True)
    u_g = env_g._denorm(a_g)
    x5, u_g = _grade_step(x5, u_g)
    env_g.state = x5.copy(); obs_g = env_g._obs()
    st_g.append(x5.copy()); _ev_grade_step(ev_g, x5, u_g, 0.)
    if float(x5[0]) >= path.s_max - 0.1 or abs(float(x5[1])) >= D_MAX - 0.1:
        break
res_grade["PPO"] = (_rmse_d(st_g), ev_g)

# PID-SF
pid_g = PIDModel(dyn_model=dyn_model, vel_profile=vel_profile, L_f=L_F,
                 t_step=T_SMPL, Kff_kappa=1.0, Kp_lon=1.0, Ki_lon=0.02)
pid_g.reset(); x5 = dyn_model.clip_state(X0.copy()); st_g = [x5.copy()]
ev_g = EVEnergyModel(T_SMPL)
for _ in range(400):
    u_g = pid_g.compute_control(x5)
    x5, u_g = _grade_step(x5, u_g)
    st_g.append(x5.copy()); _ev_grade_step(ev_g, x5, u_g, 0.)
    if float(x5[0]) >= path.s_max - 0.1 or abs(float(x5[1])) >= D_MAX - 0.1:
        break
res_grade["PID-SF"] = (_rmse_d(st_g), ev_g)

# NMPC
mpc_g = NMPCModel(dyn_model=dyn_model, vel_profile=vel_profile,
                  obj_weights=dict(obj_weights), L_f=L_F,
                  t_step=T_SMPL, n_horizon=N_HORIZON)
x5 = dyn_model.clip_state(X0.copy())
mpc_g._optimizer.x0 = x5; mpc_g._optimizer.set_initial_guess()
st_g = [x5.copy()]; ev_g = EVEnergyModel(T_SMPL)
for _ in range(300):
    t0 = time.perf_counter()
    u0 = mpc_g._optimizer.make_step(x5.reshape(-1, 1))
    te = time.perf_counter() - t0
    x5, u_g = _grade_step(x5, np.asarray(u0, float).reshape(-1))
    st_g.append(x5.copy()); _ev_grade_step(ev_g, x5, u_g, te)
    if float(x5[0]) >= path.s_max - 0.1 or abs(float(x5[1])) >= D_MAX - 0.1:
        break
res_grade["NMPC"] = (_rmse_d(st_g), ev_g)

print("=" * 70)
print(f"ROAD-GRADE ROBUSTNESS  (theta_amp = {np.rad2deg(THETA_AMP):.0f} deg, "
      "flat-road references, grade-blind controllers)")
print("=" * 70)
print(f"  {'Controller':<10} {'RMSE_d [m]':>11} {'E_net [kJ]':>11} {'Regen %':>9}"
      f"   {'Flat-road RMSE':>14}")
print("  " + "-" * 62)
_flat = {"NMPC": 0.474, "PPO": 0.314, "PID-SF": 1.155}
for c in ["NMPC", "PPO", "PID-SF"]:
    r, ev = res_grade[c]
    print(f"  {c:<10} {_f(r):>11.3f} {_f(ev.e_net)/1000:>11.2f} "
          f"{100.*_f(ev.e_recovered)/max(_f(ev.e_propulsion),1e-9):>8.1f}%"
          f"   {_flat[c]:>14.3f}")
print("=" * 70)
print("Paper: removes the flat-road limitation (Sec. 6, item on Eq. 6);")
print("grade-aware v_ref remains future work.")

In [ ]:
# Summary
print("\n" + "="*70)
print("FINAL SUMMARY")
print("="*70)
print(f"\n[Observation scaling]")
print(f"  kappa/KAPPA_SCALE in obs (kappa_max={KAPPA_MAX:.4f} m-1 -> peak maps to +/-1.0)")
print(f"  Velocity penalty normalized by v_max={V_MAX} m/s")
print(f"  Steering penalty normalized by delta_max={DELTA_MAX} rad")
print(f"\n[PID-SF Step Response]")
print(f"  Settling time (|d|<0.05m): {t_settle:.2f}s at v=5m/s")
print(f"\n[N=30 Statistical]")
for ctrl in ["NMPC","PPO","PID","STAN"]:
    r=np.array(res30[ctrl]["rmse_d"])
    e=np.array(res30[ctrl]["e_net"])
    print(f"  {ctrl:<6}: RMSE_d={np.nanmean(r):.3f}+/-{np.nanstd(r):.3f}m  "
          f"E_net={np.nanmean(e):.2f}kJ")
print(f"\n[Statistical Tests]")
print(f"  Wilcoxon (paired) p={p_wx:.4f} | Mann-Whitney p={p_mw:.4f} | Cohen's d={d_ppo_nmpc:.3f} "
      f"({effect_label(d_ppo_nmpc)})")
print(f"\n[Zero-Shot LC / Chicane]")
print(f"  LC - NMPC: {rmse_n_lc:.3f}m | PPO: {rmse_r_lc:.3f}m | PID: {rmse_p_lc:.3f}m")
print(f"  CH - NMPC: {results_zs['CH']['NMPC'][0]:.3f}m | PPO: {rmse_r_ch:.3f}m | PID: {results_zs['CH']['PID'][0]:.3f}m")
print(f"\n[Noise sigma=0.05] RMSE={noise_results[0.05]['rmse_mean']:.3f}m")
print(f"\n[Ablation]")
for name,r in ablation_results.items():
    print(f"  {name:<40}: {r['rmse_d']:.3f}m")
print("="*70)

In [ ]:
# Export figure data 
import numpy as np

def _f(x):
    """Coerce CasADi DM / numpy."""
    try:
        import casadi as _ca
        if isinstance(x, _ca.DM):
            return float(x.full().ravel()[0])
    except Exception:
        pass
    a = np.asarray(x, dtype=float).ravel()
    return float(a[0]) if a.size == 1 else float(np.nanmean(a))

def _a(x):
    return np.asarray(x, dtype=float)

export, failed = {}, []

def _try(tag, fn):
    try:
        fn()
    except Exception as exc:
        failed.append(f"{tag}: {type(exc).__name__}: {exc}")

# Fig 2
def _f2():
    s = np.linspace(0.0, float(path.s_max), 600)
    export["f2_s"]     = _a(s)
    export["f2_kappa"] = _a(path.curvature(s))
    export["f2_vref"]  = _a([vel_profile.get_maximum_speed(float(si)) for si in s])
    export["f2_smax"]  = np.array([float(path.s_max)])
    export["f2_kmax"]  = np.array([float(KAPPA_SCALE)])
_try("fig2", _f2)

# Fig 3
def _f3():
    for tag, arr in [("nmpc", states_nmpc), ("ppo", st_ppo),
                     ("pid", states_pid), ("stan", states_stan)]:
        m = np.asarray(arr, dtype=float)
        export[f"f3_t_{tag}"] = _a(np.arange(len(m)) * T_SMPL)
        export[f"f3_d_{tag}"] = _a(m[:, 1])
_try("fig3", _f3)

# Fig 5
def _f5():
    evs = [ev_nmpc, ev_ppo, ev_pid, ev_stan]
    export["f5_Ep"] = _a([_f(e.e_propulsion) / 1000 for e in evs])
    export["f5_Er"] = _a([_f(e.e_recovered) / 1000 for e in evs])
    export["f5_En"] = _a([_f(e.e_net) / 1000 for e in evs])
_try("fig5", _f5)

# Fig 7
def _f7():
    export["f7_rmse"]    = _a([_f(x) for x in rmse_all])
    export["f7_enet"]    = _a([_f(x) for x in enet_all])
    export["f7_pareto"]  = np.asarray(pareto_mask, dtype=bool)
    export["f7_valid"]   = np.asarray(v_ok, dtype=bool)
    export["f7_default"] = _a([_f(np.sqrt(np.mean(np.asarray(states_nmpc,
                                    dtype=float)[1:, 1] ** 2))),
                               _f(ev_nmpc.e_net) / 1000])
_try("fig7", _f7)

# Fig 8
def _f8():
    order = ["NMPC", "PPO", "PID", "STAN"]
    export["f8_mean"] = _a([np.nanmean([_f(x) for x in res30[c]["rmse_d"]])
                            for c in order])
    export["f8_std"]  = _a([np.nanstd([_f(x) for x in res30[c]["rmse_d"]])
                            for c in order])
    export["f8_fail"] = _a([sum(1 for x in res30[c]["fail"] if bool(x))
                            for c in order])
    export["f8_n"]    = np.array([len(res30["PPO"]["rmse_d"])])
_try("fig8", _f8)

# Fig 9
def _f9():
    export["f9_ref_wp"] = _a(WP_LC)
    lc = results_zs["LC"]
    for key, tag in [("NMPC", "nmpc"), ("PPO", "ppo"),
                     ("PID", "pid"), ("STAN", "stan")]:
        if key not in lc:
            continue
        st = np.asarray(lc[key][1], dtype=float)
        export[f"f9_s_{tag}"]    = _a(st[:, 0])
        export[f"f9_d_{tag}"]    = _a(st[:, 1])
        export[f"f9_rmse_{tag}"] = np.array([_f(lc[key][0])])
_try("fig9", _f9)

# Fig 10
def _f10():
    v = ["base", "preview", "e-reward", "preview+e"]
    export["f10_rmse"]  = _a([_f(stats_22[k][0]) for k in v])
    export["f10_regen"] = _a([_f(stats_22[k][2]) for k in v])
    export["f10_nmpc_regen"] = np.array([
        100.0 * _f(ev_nmpc.e_recovered) / max(_f(ev_nmpc.e_propulsion), 1e-9)])
    export["f10_nmpc_rmse"] = np.array([
        _f(np.sqrt(np.mean(np.asarray(states_nmpc, dtype=float)[1:, 1] ** 2)))])
_try("fig10", _f10)

# Fig 11
def _f11():
    for nm, stx in [("LQR", st_l7), ("PPO-ft", st_f7), ("PID-SF", st_p7),
                    ("PPO-zs", st_z7), ("NMPC", st_n7), ("Stanley", st_s7)]:
        m = np.asarray(stx, dtype=float)
        export[f"f11_t_{nm}"] = _a(np.arange(len(m)) * T_SMPL)
        export[f"f11_d_{nm}"] = _a(m[:, 1])
    export["f11_ok"] = np.array([1.0])
_try("fig11", _f11)
if "f11_ok" not in export:
    export["f11_ok"] = np.array([0.0])

# Figs
def _f567():
    series = [("nmpc", states_nmpc, controls_nmpc),
              ("ppo",  st_ppo,      ct_ppo),
              ("pid",  states_pid,  controls_pid),
              ("stan", states_stan, controls_stan)]
    for tag, st, ct in series:
        m = np.asarray(st, dtype=float)
        c = np.asarray(ct, dtype=float)
        n = min(len(m), len(c))
        export[f"f3_s_{tag}"]      = _a(m[:, 0])  # arclength for d-vs-s
        export[f"f56_t_{tag}"]     = _a(np.arange(len(m)) * T_SMPL)
        export[f"f56_alpha_{tag}"] = _a(m[:, 2])
        export[f"f56_u1_{tag}"]    = _a(c[:n, 0])
        export[f"f56_u2_{tag}"]    = _a(c[:n, 1])
        s_c = np.clip(m[:, 0], 0.0, float(path.s_max))
        kap = np.asarray(path.curvature(s_c), dtype=float).ravel()
        export[f"f56_alat_{tag}"]  = _a(m[:, 4] ** 2 * kap)
    export["f56_u1_max"]   = np.array([float(dyn_model.control_upper[0])])
    export["f56_u2_max"]   = np.array([float(dyn_model.control_upper[1])])
    export["f56_alat_max"] = np.array([float(A_LAT_MAX)])
_try("figs-extra", _f567)

np.savez("figure_data.npz", **export)
print(f"figure_data.npz written - {len(export)} arrays.")
if failed:
    print(f"\n{len(failed)} block(s) failed and were skipped:")
    for msg in failed:
        print("  -", msg)
else:
    print("All figure blocks exported.")